# 2 - Guide d'imputation d'observations à haute fréquence pour des données à fréquences mixtes 

Ce notebook présente les fonctionalités de la classe `HighFrequencyImputer` du module `tsforecast.frequency`, qui permet l'imputation de valeurs haute fréquence à partir de séries à basse fréquence dans des jeux de données à fréquences mixtes.

## Table des Matières

- [1 - Importation des modules](#1---importation-des-modules)
- [2 - Création des jeux de données](#2---création-des-jeux-de-données)
    - [2.1 - Jeu de données de séries temporelles](#21---jeu-de-données-de-séries-temporelles)
    - [2.2 - Jeu de données de panel](#22---jeu-de-données-de-panel)
    - [2.3 - Visualisation des données brutes](#23---visualisation-des-données-brutes)
- [3 - Classes auxiliaires du HighFrequencyImputer](#3---classes-auxiliaires-du-highfrequencyimputer)
    - [3.1 - Validation de la fréquence cible (TargetFrequencyValidator)](#31---validation-de-la-fréquence-cible-targetfrequencyvalidator)
    - [3.2 - Alignement des fréquences (FrequencyAligner)](#32---alignement-des-fréquences-frequencyaligner)
    - [3.3 - Fenêtre d'imputation (ImputationWindowCalculator)](#33---fenêtre-dimputation-imputationwindowcalculator)
- [4 - Combinaisons de keep_lower_frequencies et cascade_refitting](#4---combinaisons-de-keep_lower_frequencies-et-cascade_refitting)
    - [4.1 - Comprendre les paramètres](#41---comprendre-les-paramètres)
    - [4.2 - Scénario A : keep_lower_frequencies=False, cascade_refitting=False](#42---scénario-a--keep_lower_frequenciesfalse-cascade_refittingfalse)
    - [4.3 - Scénario B : keep_lower_frequencies=True, cascade_refitting=False](#43---scénario-b--keep_lower_frequenciestrue-cascade_refittingfalse)
    - [4.4 - Scénario C : keep_lower_frequencies=False, cascade_refitting=True](#44---scénario-c--keep_lower_frequenciesfalse-cascade_refittingtrue)
    - [4.5 - Scénario D : keep_lower_frequencies=True, cascade_refitting=True](#45---scénario-d--keep_lower_frequenciestrue-cascade_refittingtrue)
    - [4.6 - Comparaison visuelle des scénarios](#46---comparaison-visuelle-des-scénarios)
- [5 - Fenêtres d'imputation et seuil d'attrition](#5---fenêtres-dimputation-et-seuil-dattrition)
    - [5.1 - Comprendre la fenêtre P1](#51---comprendre-la-fenêtre-p1)
    - [5.2 - Les quatre scopes d'imputation](#52---les-quatre-scopes-dimputation)
    - [5.3 - Impact du seuil d'attrition](#53---impact-du-seuil-dattrition)
    - [5.4 - Imputation directe vs imputation des valeurs manquantes d'abord](#54---imputation-directe-vs-imputation-des-valeurs-manquantes-dabord)
- [6 - Impact des délais de publication sur l'imputation](#6---impact-des-délais-de-publication-sur-limputation)
    - [6.1 - Comprendre les paramètres delays et impute_delayed_values](#61---comprendre-les-paramètres-delays-et-impute_delayed_values)
    - [6.2 - Scénario sans délais (référence)](#62---scénario-sans-délais-référence)
    - [6.3 - Avec delays, impute_delayed_values=False](#63---avec-delays-impute_delayed_valuesfalse)
    - [6.4 - Sans delays, impute_delayed_values=True](#64---sans-delays-impute_delayed_valuestrue)
    - [6.5 - Avec delays, impute_delayed_values=True](#65---avec-delays-impute_delayed_valuestrue)
    - [6.6 - Comparaison des scénarios](#66---comparaison-des-scénarios)
- [7 - Intégration dans un workflow](#7---intégration-dans-un-workflow)
    - [7.1 - Traitement des délais de publication après l'imputation](#71---traitement-des-délais-de-publication-après-limputation)
    - [7.2 - Intégration dans une validation croisée](#72---intégration-dans-une-validation-croisée)
- [8 - Résumé et tableau récapitulatif](#8---résumé-et-tableau-récapitulatif)
    - [8.1 - Tableau récapitulatif des paramètres](#81---tableau-récapitulatif-des-paramètres)
    - [8.2 - Recommandations par cas d'usage](#82---recommandations-par-cas-dusage)
    - [8.3 - Points clés à retenir](#83---points-clés-à-retenir)

## 1 - Importation des modules <a id="1---importation-des-modules"></a>

Cette section présente les modules nécessaires pour utiliser le `HighFrequencyImputer`.

In [ ]:
# Importation des modules
# Modules de base
import warnings

# Manipulation de données
import numpy as np
import pandas as pd

# Graphiques
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import seaborn as sns

# Sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Module tsforecast
from tsforecast.frequency import HighFrequencyImputer
from tsforecast.frequency.target_frequency_validator import TargetFrequencyValidator
from tsforecast.frequency.frequency_aligner import FrequencyAligner
from tsforecast.frequency.imputation_window import ImputationWindowCalculator
from tsforecast.frequency.detector import detect_dataset_frequency, detect_index_frequency
from tsforecast.delays import PublicationDelayTransformer
from tsforecast.crossvals import TSOutOfSampleSplit

# Configuration de l'affichage
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore')

# Affichage
print("Modules importés avec succès !")

In [ ]:
# Fonction utilitaire d'affichage de la provenance
def display_provenance(imputer, title="Provenance des observations"):
    """Display the provenance matrix and statistics for an imputer.
    
    Args:
        imputer: Fitted HighFrequencyImputer instance.
        title: Display title.
    """
    # Récupération de la matrice de provenance
    provenance = imputer.imputation_provenance_
    
    print(f"\n{'=' * 80}")
    print(f"{title}")
    print(f"{'=' * 80}")
    
    # Conversion en chaînes lisibles
    prov_strings = provenance.map(
        lambda x: x.value if hasattr(x, 'value') else ('not_imputed' if pd.isna(x) else str(x))
    )
    
    # Affichage du comptage par type de provenance
    print("\n--- Répartition des types de provenance ---")
    for col in prov_strings.columns:
        counts = prov_strings[col].value_counts()
        print(f"\n  {col}:")
        for prov_type, count in counts.items():
            print(f"    {prov_type}: {count}")
    
    # Affichage du heatmap de provenance
    fig, ax = plt.subplots(figsize=(14, 5))
    
    # Encodage numérique de la provenance pour le heatmap
    prov_map = {
        'original': 0,
        'model_on_true': 1,
        'model_on_mixed': 2,
        'aggregated': 3,
        'not_imputed': -1
    }
    prov_numeric = prov_strings.replace(prov_map).astype(float)
    
    # Couleurs personnalisées
    from matplotlib.colors import ListedColormap, BoundaryNorm
    cmap_colors = ['#cccccc', '#2ecc71', '#3498db', '#f39c12', '#e74c3c']
    cmap = ListedColormap(cmap_colors)
    bounds = [-1.5, -0.5, 0.5, 1.5, 2.5, 3.5]
    norm = BoundaryNorm(bounds, cmap.N)
    
    im = ax.imshow(prov_numeric.T, aspect='auto', cmap=cmap, norm=norm, interpolation='nearest')
    
    # Configuration des axes
    ax.set_yticks(range(len(prov_strings.columns)))
    ax.set_yticklabels(prov_strings.columns)
    
    # Sous-ensemble des dates
    n_ticks = min(12, len(prov_strings))
    tick_indices = np.linspace(0, len(prov_strings) - 1, n_ticks, dtype=int)
    ax.set_xticks(tick_indices)
    if hasattr(prov_strings.index, 'strftime'):
        ax.set_xticklabels([prov_strings.index[i].strftime('%Y-%m') for i in tick_indices], rotation=45, ha='right')
    else:
        ax.set_xticklabels([str(prov_strings.index[i]) for i in tick_indices], rotation=45, ha='right')
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    
    # Légende
    legend_labels = ['Non imputé', 'Original', 'Model on true', 'Model on mixed', 'Agrégé']
    legend_elements = [
        mpatches.Patch(facecolor=c, label=l) for c, l in zip(cmap_colors, legend_labels)
    ]
    ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    # Affichage d'un échantillon de la matrice
    print("\n--- Aperçu de la matrice de provenance (dernières lignes) ---")
    display(prov_strings.tail(15))


## 2 - Création des jeux de données <a id="2---création-des-jeux-de-données"></a>

Nous allons créer deux jeux de données fictifs :
1. Un jeu de **séries temporelles** simples
2. Un jeu de **données de panel** (plusieurs entités)

Chaque jeu contient des variables à différentes fréquences (mensuelle, trimestrielle, annuelle) avec des délais de publication variables simulant des situations réelles.

### 2.1 - Jeu de données de séries temporelles <a id="21---jeu-de-données-de-séries-temporelles"></a>

Ce jeu de données représente des indicateurs macroéconomiques typiques d'un pays :
- **PIB** : Publication trimestrielle avec délai de 2 mois
- **Inflation (IPC)** : Publication mensuelle avec délai de 1 mois
- **Taux de chômage** : Publication mensuelle avec délai de 1 mois
- **Production industrielle** : Publication mensuelle (disponible rapidement)
- **Balance commerciale annuelle** : Publication annuelle avec délai de 3 mois

In [ ]:
# Fonction de création de séries temporelles
def create_timeseries_dataset(
    start_date: str = '2018-01-01',
    end_date: str = '2024-07-01',
    seed: int = 42
) -> pd.DataFrame:
    """Create a realistic macroeconomic time series dataset with mixed frequencies.
    
    Args:
        start_date: Start date for the dataset.
        end_date: End date for the dataset.
        seed: Random seed for reproducibility.
        
    Returns:
        DataFrame with mixed-frequency macroeconomic indicators.
    """
    # Initialisation du seed
    np.random.seed(seed)
    
    # Création de l'index mensuel
    dates = pd.date_range(start=start_date, end=end_date, freq='MS')
    n_periods = len(dates)
    
    # Initialisation du DataFrame
    df = pd.DataFrame(index=dates)
    df.index.name = 'date'
    
    # ----- Variables mensuelles -----
    # Production industrielle (mensuelle, croissance avec bruit)
    trend = np.linspace(100, 115, n_periods)
    seasonal = 3 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
    noise = np.random.normal(0, 1.5, n_periods)
    df['production_industrielle'] = trend + seasonal + noise
    
    # Inflation mensuelle (IPC, entre 0.5% et 4%)
    inflation_trend = np.linspace(1.2, 2.8, n_periods)
    inflation_noise = np.random.normal(0, 0.3, n_periods)
    df['inflation_ipc'] = np.clip(inflation_trend + inflation_noise, 0.5, 5.0)
    
    # Taux de chômage (mensuel, entre 5% et 12%)
    chomage_trend = np.concatenate([
        np.linspace(8.5, 7.0, n_periods // 3),
        np.linspace(7.0, 9.5, n_periods // 3),  # Choc économique
        np.linspace(9.5, 7.5, n_periods - 2 * (n_periods // 3))
    ])
    chomage_noise = np.random.normal(0, 0.2, n_periods)
    df['taux_chomage'] = np.clip(chomage_trend + chomage_noise, 4.0, 15.0)
    
    # ----- Variable trimestrielle : PIB -----
    # Le PIB n'est disponible qu'aux fins de trimestre
    pib_base = 2500
    pib_growth_quarterly = 0.5  # Croissance trimestrielle moyenne
    df['pib_trimestriel'] = np.nan
    
    quarter_start_months = [1, 4, 5, 10]
    quarter_idx = 0
    for i, date in enumerate(dates):
        if date.month in quarter_start_months:
            growth = pib_growth_quarterly + np.random.normal(0, 0.3)
            df.loc[date, 'pib_trimestriel'] = pib_base * (1 + growth / 100) ** quarter_idx
            quarter_idx += 1
    
    # ----- Variable annuelle : Balance commerciale -----
    df['balance_commerciale_annuelle'] = np.nan
    
    for i, date in enumerate(dates):
        if date.month == 1:
            year_factor = (date.year - 2018)
            base_balance = -25 + year_factor * 3 + np.random.normal(0, 5)
            df.loc[date, 'balance_commerciale_annuelle'] = base_balance
    
    # ----- Simulation des délais de publication -----
    # Délai de 1 mois pour l'inflation et le chômage
    df.loc[df.index[-1], 'inflation_ipc'] = np.nan
    df.loc[df.index[-1], 'taux_chomage'] = np.nan
    
    # Délai de 2 mois pour le PIB
    pib_available = df[df['pib_trimestriel'].notna()].index
    if len(pib_available) > 0:
        df.loc[pib_available[-1], 'pib_trimestriel'] = np.nan
    
    # Délai de 3 mois pour la balance commerciale annuelle
    bc_available = df[df['balance_commerciale_annuelle'].notna()].index
    if len(bc_available) > 0:
        df.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

    # ----- Simulation de données historiques limitées -----
    # La production industrielle n'est disponible qu'à partir de 2019
    mask_before_2019 = df.index < '2019-01-01'
    df.loc[mask_before_2019, 'production_industrielle'] = np.nan
    
    return df


# Création du jeu de données de séries temporelles
df_timeseries = create_timeseries_dataset()

# Affichage
print("=" * 80)
print("JEU DE DONNÉES DE SÉRIES TEMPORELLES")
print("=" * 80)
print(f"\nPériode : {df_timeseries.index.min().strftime('%Y-%m')} à {df_timeseries.index.max().strftime('%Y-%m')}")
print(f"Nombre d'observations : {len(df_timeseries)}")
print(f"Colonnes : {list(df_timeseries.columns)}")
print("\n--- Statistiques descriptives ---")
print(df_timeseries.describe().round(2))
print("\n--- Aperçu des dernières lignes ---")
display(df_timeseries.tail(15))

### 2.2 - Jeu de données de panel

<a id="22---jeu-de-données-de-panel"></a>

Ce jeu de données représente les mêmes indicateurs pour trois pays de la zone euro : France, Allemagne et Italie. Chaque pays a des caractéristiques légèrement différentes et des profondeurs historiques variables pour certaines séries.

In [ ]:
# Fonction de création d'un jeu de données de panel fictif
def create_panel_dataset(
    start_date: str = '2018-01-01',
    end_date: str = '2024-07-01',
    seed: int = 42
) -> pd.DataFrame:
    """Create a realistic macroeconomic panel dataset with mixed frequencies.
    
    Args:
        start_date: Start date for the dataset.
        end_date: End date for the dataset.
        seed: Random seed for reproducibility.
        
    Returns:
        DataFrame with MultiIndex (country, date) and mixed-frequency indicators.
    """
    # Initialisation du seed
    np.random.seed(seed)
    
    # Définition des pays et leurs caractéristiques
    countries = {
        'France': {
            'pib_base': 2800,
            'inflation_base': 1.5,
            'chomage_base': 8.0,
            'prod_ind_start': '2018-06-01'  # Historique complet
        },
        'Allemagne': {
            'pib_base': 3500,
            'inflation_base': 1.2,
            'chomage_base': 5.5,
            'prod_ind_start': '2019-01-01'  # Historique partiel
        },
        'Italie': {
            'pib_base': 2200,
            'inflation_base': 1.8,
            'chomage_base': 10.5,
            'prod_ind_start': '2019-06-01'  # Historique plus court
        }
    }
    
    # Initialisation des dates et du nombre de périodes
    dates = pd.date_range(start=start_date, end=end_date, freq='MS')
    n_periods = len(dates)
    
    # Initialisation de la liste des jeux de données pour l'ensemble des pays
    all_data = []
    
    # Parcours des pays
    for country, params in countries.items():
        np.random.seed(seed + hash(country) % 1000)
        
        # Création du DataFrame pour ce pays
        df_country = pd.DataFrame(index=dates)
        df_country['country'] = country
        
        # Production industrielle (mensuelle)
        trend = np.linspace(100, 112 + np.random.uniform(-3, 3), n_periods)
        seasonal = 2.5 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
        noise = np.random.normal(0, 1.2, n_periods)
        df_country['production_industrielle'] = trend + seasonal + noise
        
        # Données non disponibles avant une certaine date
        prod_start = pd.Timestamp(params['prod_ind_start'])
        df_country.loc[df_country.index < prod_start, 'production_industrielle'] = np.nan
        
        # Inflation (mensuelle)
        infl_trend = np.linspace(
            params['inflation_base'], 
            params['inflation_base'] + np.random.uniform(0.5, 2.0), 
            n_periods
        )
        infl_noise = np.random.normal(0, 0.25, n_periods)
        df_country['inflation_ipc'] = np.clip(infl_trend + infl_noise, 0.3, 6.0)
        
        # Taux de chômage (mensuel)
        chomage_base = params['chomage_base']
        chomage_evolution = np.concatenate([
            np.linspace(chomage_base, chomage_base - 1, n_periods // 3),
            np.linspace(chomage_base - 1, chomage_base + 2, n_periods // 3),
            np.linspace(chomage_base + 2, chomage_base + 0.5, n_periods - 2 * (n_periods // 3))
        ])
        chomage_noise = np.random.normal(0, 0.15, n_periods)
        df_country['taux_chomage'] = np.clip(chomage_evolution + chomage_noise, 2.5, 15.0)
        
        # PIB trimestriel
        df_country['pib_trimestriel'] = np.nan
        quarter_end_months = [1, 4, 7, 10]
        quarter_idx = 0
        for i, date in enumerate(dates):
            if date.month in quarter_end_months:
                growth = 0.4 + np.random.normal(0, 0.35)
                df_country.loc[date, 'pib_trimestriel'] = params['pib_base'] * (1 + growth / 100) ** quarter_idx
                quarter_idx += 1
        
        # Balance commerciale annuelle
        df_country['balance_commerciale_annuelle'] = np.nan
        for date in dates:
            if date.month == 1:
                year_factor = (date.year - 2018)
                base = -20 + np.random.uniform(-10, 10) + year_factor * 2
                df_country.loc[date, 'balance_commerciale_annuelle'] = base
        
        # Simulation des délais de publication
        df_country.loc[df_country.index[-1], 'inflation_ipc'] = np.nan
        df_country.loc[df_country.index[-1], 'taux_chomage'] = np.nan
        
        pib_available = df_country[df_country['pib_trimestriel'].notna()].index
        if len(pib_available) > 0:
            df_country.loc[pib_available[-1], 'pib_trimestriel'] = np.nan
        
        bc_available = df_country[df_country['balance_commerciale_annuelle'].notna()].index
        if len(bc_available) > 0:
            df_country.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan
        
        all_data.append(df_country)
    
    # Concaténation et création du MultiIndex
    df_panel = pd.concat(all_data, ignore_index=False)
    df_panel = df_panel.reset_index().rename(columns={'index': 'date'})
    df_panel = df_panel.set_index(['country', 'date'])
    df_panel = df_panel.sort_index()
    
    return df_panel


# Création du jeu de données de panel
df_panel = create_panel_dataset()

# Affichage
print("=" * 80)
print("JEU DE DONNÉES DE PANEL")
print("=" * 80)
print(f"\nEntités (pays) : {df_panel.index.get_level_values('country').unique().tolist()}")
print(f"Colonnes : {list(df_panel.columns)}")
print(f"Shape : {df_panel.shape}")
print("\n--- Statistiques par pays ---")
display(df_panel.groupby('country').agg(['count', 'mean']).round(2))
print("\n--- Aperçu pour la France ---")
display(df_panel.loc['France'].tail(10))

### 2.3 - Visualisation des données brutes

<a id="23---visualisation-des-données-brutes"></a>

Visualisons les données pour comprendre leur structure et la répartition des valeurs manquantes.

In [ ]:
# Fonction de visualitaion des données
def plot_data_availability(df: pd.DataFrame, title: str = "Disponibilité des données"):
    """Plot data availability heatmap showing NaN patterns.
    
    Args:
        df: DataFrame to visualize.
        title: Plot title.
    """
    # Gestion du MultiIndex pour panel
    if isinstance(df.index, pd.MultiIndex):
        plot_df = df.reset_index(level=0, drop=True)
    else:
        plot_df = df
    
    # Création de la matrice de disponibilité
    availability = plot_df.notna().astype(int)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Création du heatmap avec couleurs personnalisées
    cmap = ListedColormap(['#ffcccc', '#90EE90'])  # Rouge clair pour NaN, vert pour disponible
    
    im = ax.imshow(availability.T, aspect='auto', cmap=cmap, interpolation='nearest')
    
    # Configuration des axes
    ax.set_yticks(range(len(plot_df.columns)))
    ax.set_yticklabels(plot_df.columns)
    
    # Affichage d'un sous-ensemble des dates
    n_ticks = 12
    tick_indices = np.linspace(0, len(plot_df) - 1, n_ticks, dtype=int)
    ax.set_xticks(tick_indices)
    ax.set_xticklabels([plot_df.index[i].strftime('%Y-%m') for i in tick_indices], rotation=45, ha='right')
    
    ax.set_xlabel('Date')
    ax.set_ylabel('Variable')
    ax.set_title(title)
    
    # Légende
    legend_elements = [
        mpatches.Patch(facecolor='#90EE90', label='Données disponibles'),
        mpatches.Patch(facecolor='#ffcccc', label='Valeurs manquantes (NaN)')
    ]
    ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.02, 1))
    
    plt.tight_layout()
    plt.show()
    
    # Statistiques de disponibilité
    print("\nStatistiques de disponibilité :")
    availability_pct = (plot_df.notna().sum() / len(plot_df) * 100).round(1)
    for col, pct in availability_pct.items():
        print(f"  {col}: {pct}% disponible")


# Visualisation pour les séries temporelles
print("\n" + "=" * 80)
print("VISUALISATION DU JEU DE DONNÉES DE SÉRIES TEMPORELLES")
print("=" * 80)
plot_data_availability(df_timeseries, "Disponibilité des données - Séries temporelles")

# Visualisation pour le panel (France seulement pour simplifier)
print("\n" + "=" * 80)
print("VISUALISATION DU JEU DE DONNÉES DE PANEL (France)")
print("=" * 80)
plot_data_availability(df_panel.loc['France'], "Disponibilité des données - France (Panel)")

In [ ]:
# Fonction d'affichage des séries par fréquence
def plot_series_with_frequencies(df: pd.DataFrame, title: str = "Variables par fréquence"):
    """Plot time series colored by their frequency.
    
    Args:
        df: DataFrame to visualize.
        title: Plot title.
    """
    # Gestion du MultiIndex
    if isinstance(df.index, pd.MultiIndex):
        plot_df = df.reset_index(level=0, drop=True)
    else:
        plot_df = df
    
    # Détection automatique des fréquences
    freq_colors = {
        'Mensuelle': '#2ecc71',
        'Trimestrielle': '#3498db',
        'Annuelle': '#e74c3c'
    }
    
    fig, axes = plt.subplots(len(plot_df.columns), 1, figsize=(14, 3 * len(plot_df.columns)), sharex=True)
    if len(plot_df.columns) == 1:
        axes = [axes]
    
    for ax, col in zip(axes, plot_df.columns):
        series = plot_df[col].dropna()
        
        # Estimation de la fréquence
        if len(series) > 1:
            avg_gap = (series.index[1:] - series.index[:-1]).mean().days
            if avg_gap < 45:
                freq_label = 'Mensuelle'
            elif avg_gap < 120:
                freq_label = 'Trimestrielle'
            else:
                freq_label = 'Annuelle'
        else:
            freq_label = 'Mensuelle'
        
        color = freq_colors[freq_label]
        
        # Tracé
        ax.plot(series.index, series.values, color=color, linewidth=1.5, marker='o', markersize=3)
        ax.fill_between(series.index, series.values, alpha=0.1, color=color)
        ax.set_ylabel(col, fontsize=9)
        ax.legend([f'{freq_label}'], loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.3)
    
    axes[0].set_title(title, fontsize=12, fontweight='bold')
    axes[-1].set_xlabel('Date')
    
    plt.tight_layout()
    plt.show()


# Visualisation des séries par fréquence
plot_series_with_frequencies(df_timeseries, "Variables macroéconomiques par fréquence")

---

## 3 - Classes auxiliaires du HighFrequencyImputer

<a id="3---classes-auxiliaires-du-highfrequencyimputer"></a>

Le `HighFrequencyImputer` s'appuie sur trois classes auxiliaires pour gérer les différentes étapes du processus d'imputation :

- **`TargetFrequencyValidator`** : valide que la fréquence cible est compatible avec les fréquences détectées dans les données.
- **`FrequencyAligner`** : effectue l'agrégation (haute → basse fréquence) ou l'interpolation (basse → haute fréquence) des colonnes vers une fréquence cible.
- **`ImputationWindowCalculator`** : calcule la fenêtre temporelle où toutes les séries ont des données et gère l'extension selon le scope d'imputation.

Cette section illustre le comportement de chacune de ces classes.

### 3.1 - Validation de la fréquence cible (TargetFrequencyValidator)

<a id="31---validation-de-la-fréquence-cible-targetfrequencyvalidator"></a>

Le `TargetFrequencyValidator` vérifie que la fréquence cible demandée n'est pas plus granulaire que la fréquence la plus élevée détectée dans les données. Deux comportements sont possibles en cas d'incompatibilité :
- `on_frequency_mismatch='error'` : lève une erreur.
- `on_frequency_mismatch='warn'` : émet un avertissement et ajuste la fréquence cible.

In [ ]:
# Validation de la fréquence cible sur des données de séries temporelles
validator = TargetFrequencyValidator()

# Détection des fréquences du jeu de données de séries temporelles
detected_freqs_ts = detect_dataset_frequency(df_timeseries)
print("Fréquences détectées (séries temporelles) :")
for col, freq in detected_freqs_ts.items():
    print(f"  {col}: {freq}")

# Cas 1 : fréquence cible compatible (mensuelle ≤ mensuelle)
validated = validator.validate(
    target_frequency='MS',
    detected_frequencies=detected_freqs_ts,
)
print(f"\nFréquence cible 'MS' validée : {validated}")

# Cas 2 : fréquence cible trop élevée (journalière > mensuelle) avec warn
warnings.filterwarnings('always')
validated_adjusted = validator.validate(
    target_frequency='D',
    detected_frequencies=detected_freqs_ts,
    on_frequency_mismatch='warn',
)
print(f"\nFréquence cible 'D' ajustée à : {validated_adjusted}")
warnings.filterwarnings('ignore')

# Cas 3 : fréquence cible trop élevée avec error
try:
    validator.validate(
        target_frequency='D',
        detected_frequencies=detected_freqs_ts,
        on_frequency_mismatch='error',
    )
except ValueError as e:
    print(f"\nErreur attendue : {e}")

In [ ]:
# Validation de la fréquence cible sur des données de panel
detected_freqs_panel = detect_dataset_frequency(df_panel)

# Affichage des fréquences détectées par entité
print("Fréquences détectées (panel) :")
for key, freq in list(detected_freqs_panel.items())[:10]:
    print(f"  {key}: {freq}")
print(f"  ... ({len(detected_freqs_panel)} entrées au total)")

# Validation avec une fréquence cible unique pour toutes les entités
validated_panel = validator.validate(
    target_frequency='MS',
    detected_frequencies=detected_freqs_panel,
)
print(f"\nFréquences validées par entité : {validated_panel}")

# Validation avec un dictionnaire de fréquences cibles par entité
target_freq_dict = {
    ('France',): 'MS',
    ('Allemagne',): 'MS',
    ('Italie',): 'MS',
}
validated_panel_dict = validator.validate(
    target_frequency=target_freq_dict,
    detected_frequencies=detected_freqs_panel,
)
print(f"\nFréquences validées (dict) : {validated_panel_dict}")

### 3.2 - Alignement des fréquences (FrequencyAligner)

<a id="32---alignement-des-fréquences-frequencyaligner"></a>

Le `FrequencyAligner` permet de convertir des colonnes vers une fréquence cible :
- **Agrégation** (`aggregate_to_target`) : convertit une série haute fréquence vers une fréquence plus basse (ex. mensuel → trimestriel) par somme.
- **Interpolation** (`interpolate_to_target`) : convertit une série basse fréquence vers une fréquence plus haute (ex. trimestriel → mensuel) par interpolation linéaire.
- **Conversion automatique** (`convert_to_target`) : détecte automatiquement le sens de la conversion et applique l'agrégation ou l'interpolation.

In [ ]:
# Agrégation d'une variable mensuelle vers le trimestriel (séries temporelles)
aligner = FrequencyAligner()

# Agrégation de la production industrielle (mensuelle) vers le trimestriel
df_agg = aligner.aggregate_to_target(
    df=df_timeseries[['production_industrielle']].dropna(),
    aggregate_keys=['production_industrielle'],
    target_frequency='QS',
    is_panel=False,
)

# Comparaison avant/après
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(df_timeseries.index, df_timeseries['production_industrielle'], marker='.', markersize=3)
axes[0].set_title('Production industrielle (mensuelle)')
axes[0].set_ylabel('Valeur')

axes[1].plot(df_agg.index, df_agg['production_industrielle'], marker='o', markersize=4, color='orange')
axes[1].set_title('Production industrielle (agrégée au trimestriel)')
axes[1].set_ylabel('Valeur')

plt.suptitle('Agrégation : mensuel → trimestriel', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Observations avant agrégation : {df_timeseries['production_industrielle'].notna().sum()}")
print(f"Observations après agrégation : {df_agg['production_industrielle'].notna().sum()}")

In [ ]:
# Interpolation d'une variable trimestrielle vers le mensuel (séries temporelles)
df_interp = aligner.interpolate_to_target(
    df=df_timeseries[['pib_trimestriel']].dropna(),
    interpolate_keys=['pib_trimestriel'],
    target_frequency='MS',
    is_panel=False,
    method='linear',
)

# Comparaison avant/après
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
# Données originales (trimestrielles, avec NaN)
pib_valid = df_timeseries['pib_trimestriel'].dropna()
axes[0].plot(pib_valid.index, pib_valid.values, marker='o', markersize=5, linestyle='--')
axes[0].set_title('PIB trimestriel (original)')
axes[0].set_ylabel('Valeur')

# Données interpolées (mensuel)
axes[1].plot(df_interp.index, df_interp['pib_trimestriel'], marker='.', markersize=3, color='green')
axes[1].set_title('PIB interpolé au mensuel')
axes[1].set_ylabel('Valeur')

plt.suptitle('Interpolation : trimestriel → mensuel', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Observations avant interpolation : {pib_valid.shape[0]}")
print(f"Observations après interpolation : {df_interp['pib_trimestriel'].notna().sum()}")

In [ ]:
# Agrégation sur des données de panel (mensuel → trimestriel)
# Clés d'agrégation au format (entité, variable)
aggregate_keys_panel = [
    ('France', 'production_industrielle'),
    ('Allemagne', 'production_industrielle'),
    ('Italie', 'production_industrielle'),
]

# Fréquences cibles par entité
target_freqs = {
    ('France',): 'QS',
    ('Allemagne',): 'QS',
    ('Italie',): 'QS',
}

df_panel_agg = aligner.aggregate_to_target(
    df=df_panel,
    aggregate_keys=aggregate_keys_panel,
    target_frequency=target_freqs,
    is_panel=True,
)

# Affichage pour la France
print("Production industrielle - France (après agrégation trimestrielle) :")
france_agg = df_panel_agg.loc['France', 'production_industrielle'].dropna()
display(france_agg.tail(10))

In [ ]:
# Interpolation sur des données de panel (trimestriel → mensuel)
interpolate_keys_panel = [
    ('France', 'pib_trimestriel'),
    ('Allemagne', 'pib_trimestriel'),
    ('Italie', 'pib_trimestriel'),
]

target_freqs_interp = {
    ('France',): 'MS',
    ('Allemagne',): 'MS',
    ('Italie',): 'MS',
}

df_panel_interp = aligner.interpolate_to_target(
    df=df_panel,
    interpolate_keys=interpolate_keys_panel,
    target_frequency=target_freqs_interp,
    is_panel=True,
    method='linear',
)

# Comparaison pour l'Allemagne
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
pib_de_orig = df_panel.loc['Allemagne', 'pib_trimestriel'].dropna()
pib_de_interp = df_panel_interp.loc['Allemagne', 'pib_trimestriel'].dropna()

axes[0].plot(pib_de_orig.index, pib_de_orig.values, marker='o', markersize=5, linestyle='--')
axes[0].set_title('PIB Allemagne (trimestriel original)')

axes[1].plot(pib_de_interp.index, pib_de_interp.values, marker='.', markersize=3, color='green')
axes[1].set_title('PIB Allemagne (interpolé au mensuel)')

plt.suptitle('Panel : interpolation trimestriel → mensuel', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.3 - Fenêtre d'imputation (ImputationWindowCalculator)

<a id="33---fenêtre-dimputation-imputationwindowcalculator"></a>

L'`ImputationWindowCalculator` détermine la **fenêtre temporelle** où toutes les séries ont des données (fenêtre stricte), puis l'étend selon le paramètre `imputation_scope` et le seuil d'attrition. Il construit une matrice de couverture booléenne à la fréquence la plus élevée détectée, puis calcule l'attrition (fraction de colonnes couvertes) à chaque date.

In [ ]:
# Calcul de la fenêtre d'imputation sur des séries temporelles
calc = ImputationWindowCalculator(
    attrition_threshold=0.5,
    imputation_scope='strict',
)
calc.fit(df_timeseries)

print("=== Fenêtre d'imputation stricte (séries temporelles) ===")
print(f"Début : {calc.imputation_window_start_}")
print(f"Fin   : {calc.imputation_window_end_}")
print(f"Fréquence de l'index : {calc.index_freq_}")

# Affichage de la couverture par colonne
print("\nCouverture par colonne :")
for col, (start, end) in calc.column_coverage_.items():
    print(f"  {col}: {start} → {end}")

# Affichage de l'attrition
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(calc.attrition_by_date_.index, calc.attrition_by_date_.values, color='steelblue')
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.7, label='Attrition = 1.0 (fenêtre stricte)')
ax.axhline(y=0.5, color='orange', linestyle='--', alpha=0.7, label="Seuil d'attrition = 0.5")
ax.fill_between(
    calc.attrition_by_date_.index,
    calc.attrition_by_date_.values,
    alpha=0.2, color='steelblue'
)
ax.set_title('Attrition par date (séries temporelles)', fontweight='bold')
ax.set_ylabel('Attrition')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Comparaison des scopes d'imputation
scopes = ['strict', 'extended_backward', 'extended_forward', 'extended_both']

fig, axes = plt.subplots(len(scopes), 1, figsize=(14, 8), sharex=True)

for idx, scope in enumerate(scopes):
    calc_scope = ImputationWindowCalculator(
        attrition_threshold=0.5,
        imputation_scope=scope,
    )
    calc_scope.fit(df_timeseries)
    
    mask = calc_scope.imputation_window_mask_
    ax = axes[idx]
    
    # Attrition en arrière-plan
    ax.fill_between(
        calc_scope.attrition_by_date_.index,
        calc_scope.attrition_by_date_.values,
        alpha=0.15, color='steelblue'
    )
    
    # Fenêtre d'imputation
    mask_dates = mask.index[mask]
    if len(mask_dates) > 0:
        ax.axvspan(mask_dates.min(), mask_dates.max(), alpha=0.3, color='green', label="Fenêtre d'imputation")
    
    # Fenêtre stricte
    if calc_scope.imputation_window_start_ is not None:
        ax.axvspan(
            calc_scope.imputation_window_start_, calc_scope.imputation_window_end_,
            alpha=0.2, color='red', label='Fenêtre stricte'
        )
    
    ax.set_title(f"imputation_scope='{scope}'", fontsize=10)
    ax.set_ylabel('Attrition')
    ax.legend(loc='lower left', fontsize=8)

plt.suptitle("Impact du scope d'imputation sur la fenêtre", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Calcul de la fenêtre d'imputation sur des données de panel
calc_panel = ImputationWindowCalculator(
    attrition_threshold=0.5,
    imputation_scope='extended_both',
)
calc_panel.fit(df_panel)

print("=== Fenêtre d'imputation par entité (panel) ===")
for entity in calc_panel.imputation_window_start_:
    start = calc_panel.imputation_window_start_[entity]
    end = calc_panel.imputation_window_end_[entity]
    print(f"  {entity}: {start} → {end}")

# Affichage de l'attrition par entité
fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
for idx, (entity, attrition) in enumerate(calc_panel.attrition_by_date_.items()):
    if attrition is None:
        continue
    ax = axes[idx]
    ax.plot(attrition.index, attrition.values, color='steelblue')
    ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.7)
    ax.axhline(y=0.5, color='orange', linestyle='--', alpha=0.7)
    ax.fill_between(attrition.index, attrition.values, alpha=0.2, color='steelblue')
    
    # Fenêtre d'imputation
    mask = calc_panel.imputation_window_mask_[entity]
    if mask is not None:
        mask_dates = mask.index[mask]
        if len(mask_dates) > 0:
            ax.axvspan(mask_dates.min(), mask_dates.max(), alpha=0.2, color='green')
    
    ax.set_title(f"{entity[0]}", fontsize=10)
    ax.set_ylabel('Attrition')

plt.suptitle("Attrition par entité (panel, scope=extended_both)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 4 - Combinaisons de keep_lower_frequencies et cascade_refitting

<a id="4---combinaisons-de-keep_lower_frequencies-et-cascade_refitting"></a>

Cette section illustre l'impact des paramètres `keep_lower_frequencies` et `cascade_refitting` sur le processus d'imputation.

### 4.1 - Comprendre les paramètres <a id="41---comprendre-les-paramètres"></a>

#### `keep_lower_frequencies` (bool)

Ce paramètre contrôle la **structure de sortie** :

| Valeur | Comportement | Structure de sortie |
|--------|--------------|--------------------|
| `False` | Supprime les fréquences inférieures à la cible | Index simple `(Date)` ou `(Entity, Date)` |
| `True` | Conserve toutes les fréquences intermédiaires | MultiIndex `(Frequency, Date)` ou `(Entity, Frequency, Date)` |

#### `cascade_refitting` (bool)

Ce paramètre contrôle la **stratégie d'entraînement** :

| Valeur | Comportement | Impact |
|--------|--------------|--------|
| `False` | Un seul entraînement, prédiction directe vers la fréquence cible | Plus rapide |
| `True` | Réentraînement après chaque niveau de fréquence | Plus lent, mais utilise l'information des imputations précédentes |

#### Schéma conceptuel de l'imputation en cascade

```
┌───────────────────────────────────────────────────────────────────────────────┐
│                    IMPUTATION EN CASCADE                                      │
├───────────────────────────────────────────────────────────────────────────────┤
│                                                                               │
│  DONNÉES BRUTES                                                               │
│  ├── Annuelles (A)      ●───────────●───────────●                             │
│  ├── Trimestrielles (Q) ●──●──●──●──●──●──●──●──●                             │
│  └── Mensuelles (M)     ●●●●●●●●●●●●●●●●●●●●●●●●●                             │
│                                                                               │
│  ══════════════════════════════════════════════════════════════════════════   │
│                                                                               │
│  ÉTAPE 1: Imputation A → Q                                                    │
│  ├── Agrégation des M et Q à la fréquence A                                   │
│  ├── Entraînement du modèle sur les vraies valeurs A                          │
│  └── Prédiction des valeurs A à la fréquence Q                                │
│                                                                               │
│  ÉTAPE 2: Imputation Q → M                                                    │
│  ├── Agrégation des M à la fréquence Q                                        │
│  ├── Entraînement sur vraies Q (+ imputées si cascade_refitting=True)         │
│  └── Prédiction des valeurs Q à la fréquence M                                │
│                                                                               │
│  ══════════════════════════════════════════════════════════════════════════   │
│                                                                               │
│  SORTIE (selon keep_lower_frequencies) :                                      │
│  ├── False → Uniquement les données à fréquence M                             │
│  └── True  → Toutes les fréquences (A, Q, M) en MultiIndex                    │
│                                                                               │
└───────────────────────────────────────────────────────────────────────────────┘
```

### 4.2 - Configuraion avec keep_lower_frequencies=False, cascade_refitting=False <a id="42---scénario-a--keep_lower_frequenciesfalse-cascade_refittingfalse"></a>

**Configuration la plus simple et la plus rapide.** 

- Prédiction directe de la fréquence source vers la fréquence cible
- Pas de réentraînement intermédiaire
- Sortie uniquement à la fréquence cible


In [ ]:
# Schéma
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 4)
ax.axis('off')

# Entrée
ax.add_patch(plt.Rectangle((0.5, 3), 2, 0.8, facecolor='#3498db', alpha=0.7))
ax.text(1.5, 3.4, 'Données\nmixtes', ha='center', va='center', fontsize=10, fontweight='bold')

# Flèche
ax.annotate('', xy=(4, 3.4), xytext=(2.7, 3.4),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.text(3.35, 3.7, 'cascade_refitting=False\n(1 seul entraînement)', ha='center', fontsize=8)

# Imputation directe
ax.add_patch(plt.Rectangle((4.2, 3), 2.5, 0.8, facecolor='#e74c3c', alpha=0.7))
ax.text(5.45, 3.4, 'Imputation\ndirecte A,Q→M', ha='center', va='center', fontsize=10, fontweight='bold')

# Flèche
ax.annotate('', xy=(8, 3.4), xytext=(6.9, 3.4),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.text(7.45, 3.7, 'keep_lower_frequencies=False\n(supprime A et Q)', ha='center', fontsize=8)

# Sortie
ax.add_patch(plt.Rectangle((8.2, 3), 1.5, 0.8, facecolor='#2ecc71', alpha=0.7))
ax.text(8.95, 3.4, 'Sortie\n(M seul)', ha='center', va='center', fontsize=10, fontweight='bold')

# Titre
ax.text(5, 4.2, 'Scénario A : Imputation directe, sortie simple', ha='center', fontsize=14, fontweight='bold')

# Provenance
ax.add_patch(plt.Rectangle((1, 0.5), 8, 1.8, facecolor='#f8f9fa', edgecolor='#dee2e6', linewidth=2))
ax.text(5, 2.0, 'Provenance des valeurs imputées :', ha='center', fontsize=11, fontweight='bold')
ax.text(5, 1.5, '• Variables mensuelles : ORIGINAL (inchangées)', ha='center', fontsize=9)
ax.text(5, 1.1, '• pib_trimestriel → mensuel : MODEL_ON_TRUE (modèle entraîné sur vraies valeurs)', ha='center', fontsize=9)
ax.text(5, 0.7, '• balance_commerciale_annuelle → mensuel : MODEL_ON_TRUE', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

1. Détection des fréquences :
   - production_industrielle : Mensuelle (M) → pas d'imputation
   - inflation_ipc : Mensuelle (M) → pas d'imputation
   - taux_chomage : Mensuelle (M) → pas d'imputation
   - pib_trimestriel : Trimestrielle (Q) → imputation Q→M
   - balance_commerciale_annuelle : Annuelle (A) → imputation A→M

2. Processus d'imputation :
   - Entraînement unique sur P1 avec vraies valeurs
   - Prédiction directe : A→M et Q→M (en parallèle)
   
3. Sortie :
   - Index DatetimeIndex simple
   - Toutes les variables à fréquence mensuelle
   - Les fréquences intermédiaires ne sont PAS conservées

In [ ]:
# Configuration simple avec keep_lower_frequencies=False, cascade_refitting=False

# Initialisation de l'imputer
imputer_a = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    keep_lower_frequencies=False,
    cascade_refitting=False,
    imputation_scope='strict'
)

# Application sur les séries temporelles
df_imputed_a = imputer_a.fit_transform(df_timeseries)

# Affichage des dimensions
print(f"Shape avant imputation : {df_timeseries.shape}")
print(f"Shape après imputation : {df_imputed_a.shape}")

df_imputed_a.head()

In [ ]:
df_imputed_a.head(10)

In [ ]:
# Affichage de la provenance - Scénario A
display_provenance(imputer_a, "Provenance - Scénario A (keep_lower=False, cascade=False)")

### 4.3 - Configuration avec keep_lower_frequencies=True, cascade_refitting=False <a id="43---scénario-b--keep_lower_frequenciestrue-cascade_refittingfalse"></a>

**Conservation de toutes les fréquences intermédiaires.**

- Prédiction directe vers la fréquence cible
- Sortie avec MultiIndex incluant le niveau de fréquence

In [ ]:
# Configuration avec keep_lower_frequencies=True, cascade_refitting=False
# Initialisation de l'imputer
imputer_b = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    keep_lower_frequencies=True,
    cascade_refitting=False,
    imputation_scope='strict'
)
# Imputation
df_imputed_b = imputer_b.fit_transform(df_timeseries)

# Affichage des dimensions
print(f"Shape après imputation : {df_imputed_b.shape}")
print(f"Type d'index : {type(df_imputed_b.index).__name__}")

df_imputed_b.head()

In [ ]:
df_imputed_b.index

In [ ]:
# Affichage de la provenance - Scénario B
display_provenance(imputer_b, "Provenance - Scénario B (keep_lower=True, cascade=False)")

### 4.4 - Configuration avec keep_lower_frequencies=False, cascade_refitting=True <a id="44---scénario-c--keep_lower_frequenciesfalse-cascade_refittingtrue"></a>

**Imputation en cascade avec réentraînement.**

- Les modèles sont réentraînés après chaque niveau de fréquence, de la plus faible à la plus élevées
- Les valeurs imputées aux fréquences inférieures sont utilisées pour améliorer les prédictions suivantes
- Sortie simple à la fréquence cible uniquement


In [ ]:
# Visualisation du flux
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')

# Titre
ax.text(7, 7.5, 'Scénario C : Cascade avec réentraînement, sortie simple', 
        ha='center', fontsize=14, fontweight='bold')

# Données initiales
y_start = 6.5
ax.add_patch(plt.Rectangle((0.5, y_start), 2.5, 0.8, facecolor='#3498db', alpha=0.7, edgecolor='black'))
ax.text(1.75, y_start + 0.4, 'Données mixtes\nA + Q + M', ha='center', va='center', fontsize=9, fontweight='bold')

# Flèche vers étape 1
ax.annotate('', xy=(4.5, y_start + 0.4), xytext=(3.2, y_start + 0.4),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Étape 1 : A → Q
y_step1 = 6.5
ax.add_patch(plt.Rectangle((4.5, y_step1), 4, 0.8, facecolor='#e74c3c', alpha=0.7, edgecolor='black'))
ax.text(6.5, y_step1 + 0.4, 'Étape 1 : Imputation A → Q\nEntraînement sur vraies A', 
        ha='center', va='center', fontsize=9, fontweight='bold')

# Flèche de réentraînement
ax.annotate('', xy=(8.5, y_step1 - 0.3), xytext=(8.5, y_step1),
            arrowprops=dict(arrowstyle='->', color='#9b59b6', lw=2))
ax.text(10.5, y_step1 - 0.5, '★ Réentraînement avec\nnouvelles valeurs Q', 
        ha='center', fontsize=8, color='#9b59b6', fontweight='bold')

# Flèche vers étape 2
ax.annotate('', xy=(6.5, y_step1 - 1.2), xytext=(6.5, y_step1 - 0.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Étape 2 : Q → M
y_step2 = 4.3
ax.add_patch(plt.Rectangle((4.5, y_step2), 4, 0.8, facecolor='#f39c12', alpha=0.7, edgecolor='black'))
ax.text(6.5, y_step2 + 0.4, 'Étape 2 : Imputation Q → M\nEntraînement sur vraies+imputées Q', 
        ha='center', va='center', fontsize=9, fontweight='bold')

# Flèche vers sortie
ax.annotate('', xy=(6.5, y_step2 - 1.2), xytext=(6.5, y_step2 - 0.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Sortie
y_out = 2.1
ax.add_patch(plt.Rectangle((4.5, y_out), 4, 0.8, facecolor='#2ecc71', alpha=0.7, edgecolor='black'))
ax.text(6.5, y_out + 0.4, 'Sortie : Données mensuelles (M)\nIndex DatetimeIndex simple', 
        ha='center', va='center', fontsize=9, fontweight='bold')

# Légende provenance
ax.add_patch(plt.Rectangle((0.5, 0.3), 13, 1.5, facecolor='#f8f9fa', edgecolor='#dee2e6', linewidth=2))
ax.text(7, 1.5, 'Provenance des valeurs :', ha='center', fontsize=10, fontweight='bold')
ax.text(4, 1.0, '• Étape 1 (A→Q) : MODEL_ON_TRUE', ha='left', fontsize=9, color='#e74c3c')
ax.text(4, 0.6, '• Étape 2 (Q→M) : MODEL_ON_MIXED', ha='left', fontsize=9, color='#f39c12')
ax.text(9, 1.0, '• Variables M originales : ORIGINAL', ha='left', fontsize=9, color='#2ecc71')

plt.tight_layout()
plt.show()

```
┌─────────────────────────────────────────────────────────────────────────────┐
│  ÉTAPE 1 : Traitement des variables annuelles (A)                           │
├─────────────────────────────────────────────────────────────────────────────┤
│  1.1 Agrégation des features mensuelles et trimestrielles à A               │
│  1.2 Entraînement du modèle sur P1 (vraies valeurs A uniquement)            │
│  1.3 Prédiction A → Q (fréquence intermédiaire)                             │
│  1.4 Marquage provenance : MODEL_ON_TRUE                                    │
└─────────────────────────────────────────────────────────────────────────────┘
                                      ↓
┌─────────────────────────────────────────────────────────────────────────────┐
│  ÉTAPE 2 : Traitement des variables trimestrielles (Q)                      │
├─────────────────────────────────────────────────────────────────────────────┤
│  2.1 Agrégation des features mensuelles à Q                                 │
│  2.2 Entraînement incluant les valeurs A et celles Q imputées à l'étape 1   │
│  2.3 Prédiction Q → M (fréquence cible)                                     │
│  2.4 Marquage provenance : MODEL_ON_MIXED (car entrainé sur imputées)       │
└─────────────────────────────────────────────────────────────────────────────┘
                                      ↓
┌─────────────────────────────────────────────────────────────────────────────┐
│  SORTIE : Données à fréquence mensuelle uniquement                          │
│  (les fréquences A et Q intermédiaires sont supprimées)                     │
└─────────────────────────────────────────────────────────────────────────────┘

```

In [ ]:
# Configuration avec keep_lower_frequencies=False, cascade_refitting=True

# Initialisation de l'imputer
imputer_c = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    keep_lower_frequencies=False,
    cascade_refitting=True,
    imputation_scope='strict'
)

# Imputation
df_imputed_c = imputer_c.fit_transform(df_timeseries)

# Affichage des dimensions
print(f"Shape après imputation : {df_imputed_b.shape}")
print(f"Type d'index : {type(df_imputed_b.index).__name__}")

df_imputed_c.head()

In [ ]:
# Affichage de la provenance - Scénario C
display_provenance(imputer_c, "Provenance - Scénario C (keep_lower=False, cascade=True)")

### 4.5 - Configuration avec keep_lower_frequencies=True, cascade_refitting=True <a id="45---scénario-d--keep_lower_frequenciestrue-cascade_refittingtrue"></a>

**Configuration la plus complète**

- Imputation en cascade avec réentraînement à chaque niveau
- Conservation de toutes les fréquences intermédiaires
- Maximum d'information disponible pour l'analyse


In [ ]:
# Configuration avec keep_lower_frequencies=True, cascade_refitting=True
# Initialisation de l'imputer
imputer_d = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    keep_lower_frequencies=True,
    cascade_refitting=True,
    imputation_scope='extended_both',
    attrition_threshold=0.5
)
# Imputation
df_imputed_d = imputer_d.fit_transform(df_timeseries)

# Affichage des dimensions
print(f"Shape après imputation : {df_imputed_d.shape}")
print(f"Niveaux de l'index : {df_imputed_d.index.names}")
print(f"Fréquences disponibles : {df_imputed_d.index.get_level_values('frequency').unique().tolist()}")

# Accès à la matrice de provenance
print("\n--- Matrice de provenance ---")
display(imputer_d.imputation_provenance_.value_counts())

In [ ]:
# Affichage de la provenance - Scénario D
display_provenance(imputer_d, "Provenance - Scénario D (keep_lower=True, cascade=True)")

### 4.6 - Comparaison visuelle des scénarios

<a id="46---comparaison-visuelle-des-scénarios"></a>

Tableau comparatif des quatre configurations :

In [ ]:
# Tableau comparatif des scénarios
comparison_data = {
    'Scénario': ['A', 'B', 'C', 'D'],
    'keep_lower_frequencies': [False, True, False, True],
    'cascade_refitting': [False, False, True, True],
    'Structure sortie': [
        'Index simple (Date)',
        'MultiIndex (Freq, Date)',
        'Index simple (Date)',
        'MultiIndex (Freq, Date)'
    ],
    'Entraînements': ['1 seul', '1 seul', 'N (cascade)', 'N (cascade)'],
    'Vitesse': ['★★★★★', '★★★★☆', '★★★☆☆', '★★☆☆☆'],
    'Précision': ['★★☆☆☆', '★★☆☆☆', '★★★★☆', '★★★★★'],
    'Traçabilité': ['★★☆☆☆', '★★★★☆', '★★★☆☆', '★★★★★'],
    'Cas d\'usage': [
        'Prototypage rapide',
        'Analyse multi-échelle',
        'Production, précision',
        'Recherche complète'
    ]
}

df_comparison = pd.DataFrame(comparison_data)

print("=" * 100)
print("COMPARAISON DES SCÉNARIOS")
print("=" * 100)
display(df_comparison.set_index('Scénario'))

# Visualisation graphique
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : Trade-off Vitesse vs Précision
ax1 = axes[0]
scenarios = ['A', 'B', 'C', 'D']
speed_scores = [5, 4, 3, 2]
precision_scores = [2, 2, 4, 5]
colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']

for s, sp, pr, c in zip(scenarios, speed_scores, precision_scores, colors):
    ax1.scatter(sp, pr, s=500, c=c, alpha=0.7, edgecolors='black', linewidth=2)
    ax1.annotate(f'Scénario {s}', (sp, pr), textcoords="offset points", 
                 xytext=(10, 10), fontsize=11, fontweight='bold')

ax1.set_xlabel('Vitesse (1=lent, 5=rapide)', fontsize=12)
ax1.set_ylabel('Précision (1=basse, 5=haute)', fontsize=12)
ax1.set_title('Trade-off Vitesse vs Précision', fontsize=13, fontweight='bold')
ax1.set_xlim(1, 6)
ax1.set_ylim(1, 6)
ax1.grid(True, alpha=0.3)

# Graphique 2 : Complexité de la structure de sortie
ax2 = axes[1]
bar_positions = [0, 1, 2, 3]
structure_complexity = [1, 2, 1, 2]  # 1=simple, 2=MultiIndex
retraining = [0, 0, 1, 1]  # 0=non, 1=oui

bar_width = 0.35
bars1 = ax2.bar([p - bar_width/2 for p in bar_positions], structure_complexity, 
                bar_width, label='Complexité index', color='#3498db', alpha=0.7)
bars2 = ax2.bar([p + bar_width/2 for p in bar_positions], retraining, 
                bar_width, label='Réentraînement', color='#e74c3c', alpha=0.7)

ax2.set_xticks(bar_positions)
ax2.set_xticklabels([f'Scénario {s}' for s in scenarios])
ax2.set_ylabel('Niveau de complexité', fontsize=12)
ax2.set_title('Caractéristiques par scénario', fontsize=13, fontweight='bold')
ax2.legend()
ax2.set_ylim(0, 2.5)

plt.tight_layout()
plt.show()

---

## 5 - Fenêtres d'imputation et seuil d'attrition

<a id="5---fenêtres-dimputation-et-seuil-dattrition"></a>

Cette section explore l'impact des paramètres `imputation_scope`, `attrition_threshold` et `train_on_partial_coverage` sur le processus d'imputation.

### 5.1 - Comprendre la fenêtre stricte

<a id="51---comprendre-la-fenêtre-p1"></a>

La **fenêtre stricte d'imputation** est la période temporelle où **toutes les séries ont des valeurs réelles** (non-NaN). C'est la fenêtre de référence pour l'entraînement des modèles d'imputation.

```
Temps →          2018    2019    2020    2021    2022    2023    2024
                   |       |       |       |       |       |       |
Variable A     ────●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●○○○────
Variable B     ────○○○○●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●────
Variable C     ────○○○○○○○○●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●○○○○────
Variable D     ────●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●○○○○○○○────
                           ↑──────────────────────────↑
                           |     fenêtre stricte      |
                  Début fenêtre stricte         Fin fenêtre stricte

● = Valeur disponible
○ = Valeur manquante (NaN)
```

In [ ]:
def visualize_strict_window(df: pd.DataFrame):
    """Visualize strict window calculation on the dataset.
    
    Args:
        df: DataFrame to analyze.
    """
    # Gestion du MultiIndex
    if isinstance(df.index, pd.MultiIndex):
        plot_df = df.reset_index(level=0, drop=True)
    else:
        plot_df = df
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), height_ratios=[3, 1])
    
    # Graphique 1 : Disponibilité par variable
    ax1 = axes[0]
    
    n_cols = len(plot_df.columns)
    colors = plt.cm.tab10(np.linspace(0, 1, n_cols))
    
    for i, col in enumerate(plot_df.columns):
        # Création du masque de disponibilité
        mask = plot_df[col].notna()
        y_positions = np.where(mask, i + 0.4, np.nan)
        ax1.scatter(plot_df.index, y_positions, c=[colors[i]], s=3, alpha=0.8, label=col)
    
    # Calcul de P1 (approximatif pour illustration)
    all_available = plot_df.notna().all(axis=1)
    if all_available.any():
        p1_start = plot_df.index[all_available].min()
        p1_end = plot_df.index[all_available].max()
        
        # Zone P1
        ax1.axvspan(p1_start, p1_end, alpha=0.2, color='green', label='Fenêtre P1')
        ax1.axvline(p1_start, color='green', linestyle='--', linewidth=2, alpha=0.7)
        ax1.axvline(p1_end, color='green', linestyle='--', linewidth=2, alpha=0.7)
    
    ax1.set_yticks(range(n_cols))
    ax1.set_yticklabels(plot_df.columns)
    ax1.set_ylabel('Variables')
    ax1.set_title('Disponibilité des données et fenêtre P1', fontsize=13, fontweight='bold')
    ax1.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=8)
    
    # Graphique 2 : Taux d'attrition par date
    ax2 = axes[1]
    
    # Calcul du nombre de colonnes disponibles par date
    cols_available = plot_df.notna().sum(axis=1)
    attrition_rate = cols_available / len(plot_df.columns)
    
    ax2.fill_between(plot_df.index, attrition_rate, alpha=0.5, color='#3498db')
    ax2.plot(plot_df.index, attrition_rate, color='#2980b9', linewidth=1.5)
    
    # Lignes de seuil
    ax2.axhline(0.5, color='orange', linestyle='--', label='Seuil 50%')
    ax2.axhline(0.7, color='red', linestyle='--', label='Seuil 70%')
    ax2.axhline(1.0, color='green', linestyle='--', label='100% (P1)')
    
    ax2.set_ylabel('Taux de\ndisponibilité')
    ax2.set_xlabel('Date')
    ax2.set_ylim(0, 1.1)
    ax2.legend(loc='lower right', fontsize=8)
    ax2.set_title('Taux de couverture des données par date', fontsize=11)
    
    plt.tight_layout()
    plt.show()
    
    # Statistiques
    if all_available.any():
        print(f"\n--- Statistiques de la fenêtre P1 ---")
        print(f"Début P1 : {p1_start.strftime('%Y-%m')}")
        print(f"Fin P1 : {p1_end.strftime('%Y-%m')}")
        print(f"Durée P1 : {all_available.sum()} observations")
    else:
        print("\n⚠️ Aucune période avec 100% de données disponibles !")


# Visualisation de la fenêtre stricte
print("=" * 80)
print("VISUALISATION DE LA FENÊTRE STRICTE")
print("=" * 80)
visualize_strict_window(df_timeseries)

#### Exemple avec `imputation_scope='strict'`

L'imputation en mode `strict` n'utilise que la fenêtre où toutes les séries ont des vraies valeurs pour l'entraînement du modèle.

In [ ]:
# Exemple d'imputation avec scope strict
imputer_strict = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    keep_lower_frequencies=False,
    cascade_refitting=False,
    imputation_scope='strict'
)

# Imputation
df_strict = imputer_strict.fit_transform(df_timeseries)

# Affichage du résultat
print(f"Shape après imputation : {df_strict.shape}")
print(f"Fenêtre P1 : {imputer_strict.p1_window_}")
print(f"Fenêtre d'entraînement : {imputer_strict.training_window_}")
display(df_strict.tail(10))

# Affichage de la provenance
display_provenance(imputer_strict, "Provenance - imputation_scope='strict'")

### 5.2 - Les quatre scopes d'imputation

<a id="52---les-quatre-scopes-dimputation"></a>

Le paramètre `imputation_scope` contrôle comment la fenêtre d'imputation est étendue pour l'entraînement des modèles :

| Scope | Description | Fenêtre d'entraînement |
|-------|-------------|------------------------|
| `strict` | Utilise uniquement la fenêtre stricte | [Début fenêtre stricte, Fin fenêtre stricte] |
| `extended_backward` | Étend la fenêtre stricte vers le passé | [Début étendu, Fin fenêtre stricte] |
| `extended_forward` | Étend la fenêtre stricte vers le futur | [Début fenêtre stricte, Fin étendue] |
| `extended_both` | Étend dans les deux directions | [Début étendu, Fin étendue] |

In [ ]:
def visualize_imputation_scopes():
    """Create a visual diagram of the four imputation scopes."""
    fig, axes = plt.subplots(4, 1, figsize=(14, 10))
    
    scopes = [
        ('strict', 'Uniquement la fenêtre P1'),
        ('extended_backward', 'P1 + Extension vers le passé'),
        ('extended_forward', 'P1 + Extension vers le futur'),
        ('extended_both', 'P1 + Extension dans les deux sens')
    ]
    
    for ax, (scope, title) in zip(axes, scopes):
        ax.set_xlim(0, 10)
        ax.set_ylim(0, 1)
        ax.axis('off')
        
        # Timeline de base
        ax.axhline(0.5, color='black', linewidth=2, xmin=0.05, xmax=0.95)
        
        # Marqueurs temporels
        for x, label in [(1, '2018'), (2.5, '2019'), (4, 'P1\nstart'), (6, 'P1\nend'), (7.5, '2023'), (9, '2024')]:
            ax.plot(x, 0.5, 'ko', markersize=6)
            ax.text(x, 0.3, label, ha='center', fontsize=8)
        
        # Zone P1 (toujours présente)
        ax.add_patch(plt.Rectangle((4, 0.45), 2, 0.1, facecolor='#2ecc71', alpha=0.8))
        ax.text(5, 0.7, 'P1', ha='center', fontsize=10, fontweight='bold', color='#27ae60')
        
        # Extensions selon le scope
        if scope == 'extended_backward' or scope == 'extended_both':
            # Extension vers le passé
            ax.add_patch(plt.Rectangle((2, 0.45), 2, 0.1, facecolor='#f39c12', alpha=0.6))
            ax.annotate('', xy=(2, 0.5), xytext=(4, 0.5),
                       arrowprops=dict(arrowstyle='<->', color='#e67e22', lw=2))
            ax.text(3, 0.75, 'Extension\n(avant)', ha='center', fontsize=8, color='#d35400')
        
        if scope == 'extended_forward' or scope == 'extended_both':
            # Extension vers le futur
            ax.add_patch(plt.Rectangle((6, 0.45), 1.5, 0.1, facecolor='#e74c3c', alpha=0.6))
            ax.annotate('', xy=(6, 0.5), xytext=(7.5, 0.5),
                       arrowprops=dict(arrowstyle='<->', color='#c0392b', lw=2))
            ax.text(6.75, 0.75, 'Extension\n(après)', ha='center', fontsize=8, color='#a93226')
        
        # Titre
        ax.text(0.5, 0.9, f"{scope}", fontsize=11, fontweight='bold', transform=ax.transAxes)
        ax.text(0.5, 0.05, title, fontsize=9, ha='center', transform=ax.transAxes, style='italic')
    
    plt.suptitle('Les quatre scopes d\'imputation', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


print("=" * 80)
print("LES QUATRE FENETRES D'IMPUTATION")
print("=" * 80)
visualize_imputation_scopes()

#### Exemples comparatifs des quatre scopes

Comparons les résultats d'imputation avec chacun des quatre scopes d'imputation.

In [ ]:
# Comparaison des quatre scopes d'imputation
scopes = ['strict', 'extended_backward', 'extended_forward', 'extended_both']
imputers_scope = {}
results_scope = {}

for scope in scopes:
    # Initialisation de l'imputer
    imp = HighFrequencyImputer(
        target_frequency='M',
        estimator=LinearRegression(),
        keep_lower_frequencies=False,
        cascade_refitting=False,
        imputation_scope=scope,
        attrition_threshold=0.5
    )
    # Imputation
    results_scope[scope] = imp.fit_transform(df_timeseries)
    imputers_scope[scope] = imp
    
    # Affichage de la fenêtre d'entraînement
    print(f"\n--- Scope: {scope} ---")
    print(f"  Fenêtre P1 : {imp.p1_window_}")
    print(f"  Fenêtre d'entraînement : {imp.training_window_}")

# Affichage des résultats et provenances
for scope in scopes:
    display_provenance(imputers_scope[scope], f"Provenance - imputation_scope='{scope}'")

### 5.3 - Impact du seuil d'attrition

<a id="53---impact-du-seuil-dattrition"></a>

Le paramètre `attrition_threshold` (entre 0 et 1) définit le pourcentage minimum de colonnes qui doivent avoir des données pour qu'une date soit incluse dans la fenêtre d'entraînement étendue.

**Exemples :**
- `attrition_threshold=0.5` : Au moins 50% des colonnes doivent avoir des données
- `attrition_threshold=0.8` : Au moins 80% des colonnes doivent avoir des données
- `attrition_threshold=1.0` : Équivalent à `strict` (100% requis)

In [ ]:
def demonstrate_attrition_impact(df: pd.DataFrame):
    """Demonstrate the impact of different attrition thresholds.
    
    Args:
        df: DataFrame to analyze.
    """
    # Gestion du MultiIndex
    if isinstance(df.index, pd.MultiIndex):
        plot_df = df.reset_index(level=0, drop=True)
    else:
        plot_df = df
    
    # Calcul du taux de couverture par date
    coverage_rate = plot_df.notna().sum(axis=1) / len(plot_df.columns)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    thresholds = [0.3, 0.5, 0.7, 0.9]
    
    for ax, threshold in zip(axes.flat, thresholds):
        # Identification des périodes valides
        valid_mask = coverage_rate >= threshold
        
        # Tracé de la couverture
        ax.fill_between(plot_df.index, coverage_rate, alpha=0.3, color='#3498db')
        ax.plot(plot_df.index, coverage_rate, color='#2980b9', linewidth=1.5, label='Couverture')
        
        # Mise en évidence des périodes valides
        ax.fill_between(plot_df.index, 0, 1, where=valid_mask, 
                       alpha=0.2, color='green', label='Fenêtre valide')
        
        # Seuil
        ax.axhline(threshold, color='red', linestyle='--', linewidth=2, 
                  label=f'Seuil = {threshold:.0%}')
        
        # Statistiques
        n_valid = valid_mask.sum()
        pct_valid = n_valid / len(plot_df) * 100
        
        ax.set_title(f'attrition_threshold = {threshold}\n({n_valid} obs. valides, {pct_valid:.1f}%)', 
                    fontsize=11, fontweight='bold')
        ax.set_ylabel('Taux de couverture')
        ax.set_ylim(0, 1.1)
        ax.legend(loc='lower right', fontsize=8)
        ax.grid(True, alpha=0.3)
    
    plt.suptitle('Impact du seuil d\'attrition sur la fenêtre d\'entraînement', 
                fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Tableau récapitulatif
    print("\n--- Récapitulatif par seuil ---")
    summary_data = []
    for threshold in [0.3, 0.5, 0.7, 0.9, 1.0]:
        valid = coverage_rate >= threshold
        if valid.any():
            start = plot_df.index[valid].min().strftime('%Y-%m')
            end = plot_df.index[valid].max().strftime('%Y-%m')
        else:
            start = end = 'N/A'
        summary_data.append({
            'Seuil': f'{threshold:.0%}',
            'Observations valides': valid.sum(),
            'Début': start,
            'Fin': end
        })
    
    display(pd.DataFrame(summary_data))


print("=" * 80)
print("IMPACT DU SEUIL D'ATTRITION")
print("=" * 80)
demonstrate_attrition_impact(df_timeseries)

#### Exemples avec différents seuils d'attrition

Comparons l'impact de différents seuils d'attrition sur l'imputation avec le scope `extended_both`.

In [ ]:
# Comparaison de différents seuils d'attrition
thresholds = [0.3, 0.5, 0.7, 0.9]
imputers_threshold = {}
results_threshold = {}

for threshold in thresholds:
    # Initialisation de l'imputer
    imp = HighFrequencyImputer(
        target_frequency='M',
        estimator=LinearRegression(),
        keep_lower_frequencies=False,
        cascade_refitting=False,
        imputation_scope='extended_both',
        attrition_threshold=threshold
    )
    # Imputation
    results_threshold[threshold] = imp.fit_transform(df_timeseries)
    imputers_threshold[threshold] = imp
    
    # Affichage de la fenêtre d'entraînement
    print(f"\n--- Seuil d'attrition: {threshold} ---")
    print(f"  Fenêtre P1 : {imp.p1_window_}")
    print(f"  Fenêtre d'entraînement : {imp.training_window_}")

# Affichage des provenances pour les seuils extrêmes
display_provenance(imputers_threshold[0.3], "Provenance - attrition_threshold=0.3")
display_provenance(imputers_threshold[0.9], "Provenance - attrition_threshold=0.9")


### 5.4 - Imputation directe vs imputation des valeurs manquantes d'abord <a id="54---imputation-directe-vs-imputation-des-valeurs-manquantes-dabord"></a>

Le paramètre `train_on_partial_coverage` contrôle si les valeurs imputées sont utilisées pour l'entraînement des modèles **en dehors de la fenêtre d'imputation stricte**.

| `train_on_partial_coverage` | Comportement |
|---------------------------|---------------|
| `False` | Utilise uniquement les vraies valeurs pour l'entraînement |
| `True` | Utilise vraies valeurs + valeurs imputées précédemment |

```
┌─────────────────────────────────────────────────────────────────────────────┐
│  COMPARAISON DES STRATÉGIES D'ENTRAÎNEMENT                                  │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  train_on_partial_coverage = False                                          │
│  ─────────────────────────────────                                          │
│  ✓ Plus conservateur et prudent                                             │
│  ✓ Pas de propagation d'erreurs d'imputation                                │
│  ✗ Moins de données d'entraînement                                          │
│  ✗ Peut être insuffisant si la fenêtre d'imputation stricte est courte      │
│                                                                             │
│  → Recommandé quand : la fenêtre d'imputation stricte                       │
│                       est suffisamment longue, données de haute qualité     │
│                                                                             │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  train_on_partial_coverage = True                                           │
│  ────────────────────────────────                                           │
│  ✓ Plus de données d'entraînement                                           │
│  ✓ Meilleure couverture des patterns temporels                              │
│  ✗ Risque de propagation d'erreurs                                          │
│  ✗ Provenance mixte des valeurs (MODEL_ON_MIXED)                            │
│                                                                             │
│  → Recommandé quand : la fenêtre d'imputation stricte est courte,           │
│                       besoin de plus de données                             │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
def visualize_training_strategies():
    """Visualize the two training strategies."""
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Stratégie 1 : train_on_partial_coverage=False
    ax1 = axes[0]
    ax1.set_xlim(0, 10)
    ax1.set_ylim(0, 6)
    ax1.axis('off')
    ax1.set_title('train_on_partial_coverage = False\n(Entraînement sur vraies valeurs uniquement)', 
                 fontsize=11, fontweight='bold')
    
    # Données d'entrée
    ax1.add_patch(plt.Rectangle((0.5, 4.5), 9, 1, fc='#ecf0f1', ec='black'))
    ax1.text(5, 5, 'Données avec NaN', ha='center', va='center', fontsize=10)
    
    # Flèche
    ax1.annotate('', xy=(5, 3.8), xytext=(5, 4.3),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    # Séparation P1 / hors P1
    ax1.add_patch(plt.Rectangle((0.5, 2.5), 4, 1, fc='#2ecc71', alpha=0.6, ec='black'))
    ax1.text(2.5, 3, 'P1\n(vraies valeurs)', ha='center', va='center', fontsize=9)
    
    ax1.add_patch(plt.Rectangle((4.5, 2.5), 5, 1, fc='#e74c3c', alpha=0.3, ec='black'))
    ax1.text(7, 3, 'Hors P1\n(non utilisé)', ha='center', va='center', fontsize=9, color='gray')
    
    # Flèche
    ax1.annotate('', xy=(5, 1.8), xytext=(5, 2.3),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    # Entraînement
    ax1.add_patch(plt.Rectangle((2, 0.5), 6, 1, fc='#3498db', alpha=0.7, ec='black'))
    ax1.text(5, 1, 'Modèle entraîné sur P1 uniquement', ha='center', va='center', fontsize=9, fontweight='bold')
    
    # Stratégie 2 : train_on_partial_coverage=True
    ax2 = axes[1]
    ax2.set_xlim(0, 10)
    ax2.set_ylim(0, 6)
    ax2.axis('off')
    ax2.set_title('train_on_partial_coverage = True\n(Entraînement sur vraies + imputées)', 
                 fontsize=11, fontweight='bold')
    
    # Données d'entrée
    ax2.add_patch(plt.Rectangle((0.5, 4.5), 9, 1, fc='#ecf0f1', ec='black'))
    ax2.text(5, 5, 'Données avec NaN', ha='center', va='center', fontsize=10)
    
    # Flèche
    ax2.annotate('', xy=(5, 3.8), xytext=(5, 4.3),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    # Séparation P1 / hors P1 (les deux utilisés)
    ax2.add_patch(plt.Rectangle((0.5, 2.5), 4, 1, fc='#2ecc71', alpha=0.6, ec='black'))
    ax2.text(2.5, 3, 'P1\n(vraies valeurs)', ha='center', va='center', fontsize=9)
    
    ax2.add_patch(plt.Rectangle((4.5, 2.5), 5, 1, fc='#f39c12', alpha=0.6, ec='black'))
    ax2.text(7, 3, 'Hors P1\n(imputées)', ha='center', va='center', fontsize=9)
    
    # Flèches de fusion
    ax2.annotate('', xy=(5, 1.8), xytext=(2.5, 2.3),
                arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2))
    ax2.annotate('', xy=(5, 1.8), xytext=(7, 2.3),
                arrowprops=dict(arrowstyle='->', color='#e67e22', lw=2))
    
    # Entraînement
    ax2.add_patch(plt.Rectangle((2, 0.5), 6, 1, fc='#9b59b6', alpha=0.7, ec='black'))
    ax2.text(5, 1, 'Modèle entraîné sur P1 + imputées', ha='center', va='center', fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    

print("=" * 80)
print("STRATÉGIES D'ENTRAÎNEMENT")
print("=" * 80)
visualize_training_strategies()

#### Exemples comparatifs de `train_on_partial_coverage`

Comparons le comportement de l'imputer avec et sans entraînement sur les valeurs partiellement couvertes.

In [ ]:
# Comparaison train_on_partial_coverage=False vs True

# Configuration 1 : train_on_partial_coverage=False
imputer_no_partial = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    keep_lower_frequencies=False,
    cascade_refitting=True,
    imputation_scope='extended_both',
    attrition_threshold=0.5,
    train_on_partial_coverage=False
)
df_no_partial = imputer_no_partial.fit_transform(df_timeseries)

# Configuration 2 : train_on_partial_coverage=True
imputer_with_partial = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    keep_lower_frequencies=False,
    cascade_refitting=True,
    imputation_scope='extended_both',
    attrition_threshold=0.5,
    train_on_partial_coverage=True
)
df_with_partial = imputer_with_partial.fit_transform(df_timeseries)

# Affichage des fenêtres
print("--- train_on_partial_coverage=False ---")
print(f"  Fenêtre P1 : {imputer_no_partial.p1_window_}")
print(f"  Fenêtre d'entraînement : {imputer_no_partial.training_window_}")

print("\n--- train_on_partial_coverage=True ---")
print(f"  Fenêtre P1 : {imputer_with_partial.p1_window_}")
print(f"  Fenêtre d'entraînement : {imputer_with_partial.training_window_}")

# Affichage des résultats
print("\n--- Dernières valeurs (train_on_partial_coverage=False) ---")
display(df_no_partial.tail(10))
print("\n--- Dernières valeurs (train_on_partial_coverage=True) ---")
display(df_with_partial.tail(10))

# Affichage des provenances
display_provenance(imputer_no_partial, "Provenance - train_on_partial_coverage=False")
display_provenance(imputer_with_partial, "Provenance - train_on_partial_coverage=True")

## 6 - Impact des délais de publication sur l'imputation

<a id="6---impact-des-délais-de-publication-sur-limputation"></a>

Cette section détaille l'interaction entre les paramètres `delays` et `impute_delayed_values` du `HighFrequencyImputer` et le reste de la logique implémentée dans ce transformer.

Le `HighFrequencyImputer` offre deux mécanismes complémentaires pour gérer ces délais :
- **`delays`** : un DataFrame décrivant les délais de publication par variable. Il permet à l'imputer de distinguer les valeurs structurellement manquantes (pas encore publiées) des valeurs réellement absentes.
- **`impute_delayed_values`** : un booléen contrôlant si les valeurs affectées par les délais de publication doivent être imputées par le modèle.

### 6.1 - Comprendre les paramètres `delays` et `impute_delayed_values` <a id="61---comprendre-les-paramètres-delays-et-impute_delayed_values"></a>

#### `delays` (DataFrame | None)

Ce paramètre accepte un DataFrame décrivant les délais de publication de chaque variable. Les colonnes requises sont :

| Colonne | Description | Exemple |
|---------|-------------|---------|
| `variable` | Nom de la variable concernée | `'pib_trimestriel'` |
| `delay` | Valeur du délai | `45` |
| `unit` | Unité du délai | `'D'` (jours) |
| `reference_point` | Point de référence du délai | `'end'` (fin de période) |

Lorsque `delays` est fourni, l'imputer en tient compte pour le calcul de la fenêtre d'imputation. Les délais sont de plus inversés de manière à aligner dans le temps les différentes observations (on fait l'hypothèse que ce problème de prédiction est plus simple)

#### `impute_delayed_values` (bool)

| Valeur | Comportement |
|--------|-------------|
| `False` (défaut) | Les valeurs affectées par les délais de publication restent `NaN` après imputation |
| `True` | Les valeurs affectées par les délais sont imputées par le modèle, comme les autres valeurs manquantes |

#### Combinaisons possibles

| `delays` | `impute_delayed_values` | Comportement |
|----------|------------------------|-------------|
| `None` | `False` | Aucune gestion des délais. Les `NaN` en fin de série sont traités comme des données manquantes ordinaires |
| DataFrame | `False` | Les délais sont pris en compte pour la définition de la fenêtre d'imputation et sont inversés, mais les valeurs retardées ne sont **pas** imputées |
| `None` | `True` | Les délais sont **inférés** depuis les patterns de `NaN` en fin de série, puis les valeurs retardées sont imputées |
| DataFrame | `True` | Les délais spécifiés sont utilisés pour identifier et imputer les valeurs retardées |

### 6.2 - Scénario sans délais (référence) <a id="62---scénario-sans-délais-référence"></a>

Commençons par un scénario de référence sans aucune gestion des délais, pour observer le comportement par défaut de l'imputer face aux `NaN` en fin de série.

In [ ]:
# Scénario de référence : aucune gestion des délais
imputer_no_delays = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    keep_lower_frequencies=False,
    cascade_refitting=True,
    imputation_scope='strict',
    delays=None,
    impute_delayed_values=False
)

# Application sur les séries temporelles
df_no_delays = imputer_no_delays.fit_transform(df_timeseries)

# Affichage
print("=== Scénario de référence (sans gestion des délais) ===")
print(f"Shape : {df_no_delays.shape}")
print(f"\nFenêtre P1 : {imputer_no_delays.p1_window_}")
print("\n--- Dernières lignes ---")
display(df_no_delays.tail(8))

Sans gestion des délais, les `NaN` en fin de série sont traités comme des données manquantes ordinaires. L'imputer ne fait pas la distinction entre une valeur structurellement absente (pas encore publiée) et une valeur véritablement manquante.

### 6.3 - Avec delays, impute_delayed_values=False <a id="63---avec-delays-impute_delayed_valuesfalse"></a>

En fournissant un DataFrame de délais **sans** activer l'imputation des valeurs retardées, on informe l'imputer de la structure des délais. Cela peut modifier le calcul de la fenêtre d'imputation car l'imputer sait que certains `NaN` en fin de série sont attendus. Cela permet également l'imputer de traiter un problème de prévision plus simple en alignant les observations se référant à une même période.

In [ ]:
# Définition des délais de publication
delays_df = pd.DataFrame({
    'variable': ['inflation_ipc', 'taux_chomage', 'pib_trimestriel', 'balance_commerciale_annuelle'],
    'delay': [30, 30, 60, 90],
    'unit': ['D', 'D', 'D', 'D'],
    'reference_point': ['end', 'end', 'end', 'end']
})

# Affichage
print("=== Délais de publication ===")
display(delays_df)

In [ ]:
# Scénario avec delays, impute_delayed_values=False
imputer_delays_no_impute = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    keep_lower_frequencies=False,
    cascade_refitting=True,
    imputation_scope='strict',
    delays=delays_df,
    impute_delayed_values=False
)

# Application
df_delays_no_impute = imputer_delays_no_impute.fit_transform(df_timeseries)

# Comparaison des fenêtres P1
print("=== Avec delays, impute_delayed_values=False ===")
print(f"Shape : {df_delays_no_impute.shape}")
print(f"\nFenêtre P1 : {imputer_delays_no_impute.p1_window_}")
print(f"Fenêtre P1 (référence sans délais) : {imputer_no_delays.p1_window_}")
print("\n--- Dernières lignes ---")
display(df_delays_no_impute.tail(8))

En fournissant les délais, l'imputer peut ajuster sa fenêtre d'imputation. Les valeurs retardées restent `NaN` après imputation puisque `impute_delayed_values=False`. Ce mode est utile lorsqu'on souhaite traiter les délais de publication séparément, par exemple avec un `PublicationDelayTransformer` en aval.

### 6.4 - Sans delays, impute_delayed_values=True <a id="64---sans-delays-impute_delayed_valuestrue"></a>

Lorsque `impute_delayed_values=True` sans fournir de DataFrame de délais, l'imputer tente d'**inférer** les délais à partir des patterns de `NaN` en fin de série. Il identifie les colonnes ayant des `NaN` consécutifs à la fin et estime le nombre de périodes de retard.

In [ ]:
# Scénario sans delays, impute_delayed_values=True (inférence automatique)
imputer_infer_delays = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    keep_lower_frequencies=False,
    cascade_refitting=True,
    imputation_scope='strict',
    delays=None,
    impute_delayed_values=True
)

# Application
df_infer_delays = imputer_infer_delays.fit_transform(df_timeseries)

# Affichage des délais inférés
print("=== Sans delays, impute_delayed_values=True (inférence) ===")
print(f"Shape : {df_infer_delays.shape}")
print(f"\nFenêtre P1 : {imputer_infer_delays.p1_window_}")
print(f"\nDélais inférés : {imputer_infer_delays.inferred_delays_}")
print("\n--- Dernières lignes ---")
display(df_infer_delays.tail(8))

L'inférence automatique fonctionne bien lorsque les délais se traduisent par des `NaN` consécutifs en fin de série. Cependant, elle peut être imprécise si les `NaN` en fin de série ont d'autres causes (données structurellement manquantes, par exemple). C'est pourquoi il est recommandé de fournir explicitement un DataFrame de délais lorsque ceux-ci sont connus.

### 6.5 - Avec delays, impute_delayed_values=True <a id="65---avec-delays-impute_delayed_valuestrue"></a>

La combinaison la plus complète : les délais sont explicitement spécifiés **et** les valeurs retardées sont imputées. L'imputer utilise les délais fournis pour identifier précisément les observations affectées, puis impute ces valeurs avec les modèles entraînés.

In [ ]:
# Scénario avec delays et impute_delayed_values=True
imputer_delays_impute = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    keep_lower_frequencies=False,
    cascade_refitting=True,
    imputation_scope='strict',
    delays=delays_df,
    impute_delayed_values=True
)

# Application
df_delays_impute = imputer_delays_impute.fit_transform(df_timeseries)

# Affichage
print("=== Avec delays, impute_delayed_values=True ===")
print(f"Shape : {df_delays_impute.shape}")
print(f"\nFenêtre P1 : {imputer_delays_impute.p1_window_}")
print("\n--- Dernières lignes ---")
display(df_delays_impute.tail(8))

### 6.6 - Comparaison des scénarios <a id="66---comparaison-des-scénarios"></a>

Comparons visuellement les résultats des quatre scénarios sur les dernières observations, là où les délais de publication ont le plus d'impact.

In [ ]:
# Comparaison des quatre scénarios de gestion des délais
scenarios = {
    'Référence (sans délais)': df_no_delays,
    'delays, impute=False': df_delays_no_impute,
    'Inférence, impute=True': df_infer_delays,
    'delays, impute=True': df_delays_impute
}

# Sélection d'une variable représentative pour la comparaison
variable = 'inflation_ipc'

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey=True)
fig.suptitle(f"Impact des délais de publication sur l'imputation de '{variable}'", 
             fontsize=14, fontweight='bold')

for ax, (name, df_result) in zip(axes.flatten(), scenarios.items()):
    # Tracé des dernières 24 observations
    df_plot = df_result[variable].tail(24)
    
    # Données originales
    df_orig = df_timeseries[variable].loc[df_plot.index]
    
    # Identification des valeurs imputées (NaN dans l'original, non-NaN dans le résultat)
    is_imputed = df_orig.isna() & df_plot.notna()
    is_missing = df_plot.isna()
    
    # Tracé
    ax.plot(df_plot.index, df_plot.values, 'b-o', markersize=4, label='Valeur finale', zorder=2)
    ax.plot(df_orig.dropna().index, df_orig.dropna().values, 'g^', markersize=6, 
            label='Original', alpha=0.7, zorder=3)
    
    # Mise en évidence des imputations
    if is_imputed.any():
        ax.scatter(df_plot.index[is_imputed], df_plot[is_imputed].values, 
                   c='orange', s=60, marker='s', label='Imputé', zorder=4)
    
    # Mise en évidence des NaN restants
    if is_missing.any():
        for idx in df_plot.index[is_missing]:
            ax.axvline(x=idx, color='red', alpha=0.3, linestyle='--')
    
    ax.set_title(name, fontsize=11)
    ax.legend(fontsize=8, loc='upper left')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Tableau comparatif des scénarios de gestion des délais
print("=" * 100)
print("COMPARAISON DES SCÉNARIOS DE GESTION DES DÉLAIS")
print("=" * 100)

# Comptage des NaN dans les dernières lignes pour chaque scénario
n_tail = 6
print(f"\nNombre de NaN dans les {n_tail} dernières observations :")
print("-" * 80)
print(f"{'Scénario':<35} ", end="")
for col in df_timeseries.columns:
    print(f"{col[:15]:>16}", end="")
print()
print("-" * 80)

for name, df_result in scenarios.items():
    tail = df_result.tail(n_tail)
    print(f"{name:<35} ", end="")
    for col in df_result.columns:
        n_nan = tail[col].isna().sum()
        print(f"{n_nan:>16}", end="")
    print()

print("-" * 80)
print(f"{'Données originales':<35} ", end="")
orig_tail = df_timeseries.tail(n_tail)
for col in df_timeseries.columns:
    n_nan = orig_tail[col].isna().sum()
    print(f"{n_nan:>16}", end="")
print()

**Points clés de la comparaison :**

- **Sans gestion des délais** : les `NaN` en fin de série peuvent impacter le calcul de la fenêtre d'imputation et les valeurs retardées ne sont pas distinguées des données structurellement manquantes.
- **Avec `delays` seul** : la fenêtre d'imputation est mieux calibrée car l'imputer sait que les `NaN` récents sont attendus. Les valeurs retardées restent `NaN`, ce qui est utile si elles seront traitées séparément (par ex. avec un `PublicationDelayTransformer`).
- **Avec `impute_delayed_values=True`** : les valeurs retardées sont imputées, ce qui produit un jeu de données complet. L'inférence automatique est pratique mais moins précise que la spécification explicite des délais.

---

## 7 - Intégration dans un workflow

<a id="7---intégration-dans-un-workflow"></a>

Cette section montre comment intégrer le `HighFrequencyImputer` dans un workflow complet, en combinaison avec le `PublicationDelayTransformer` et les splitters de validation croisée du package. On y aborde deux cas d'usage courants :

1. **Traitement des délais de publication après l'imputation** avec le `PublicationDelayTransformer`
2. **Intégration dans une validation croisée temporelle** avec gestion rigoureuse du leakage temporel

### 7.1 - Traitement des délais de publication après l'imputation <a id="71---traitement-des-délais-de-publication-après-limputation"></a>

Dans un workflow de prévision, il est souvent préférable de séparer l'imputation des fréquences mixtes et la gestion des délais de publication en deux étapes distinctes :

1. **Étape 1** : Imputation avec `HighFrequencyImputer` (sans imputer les valeurs retardées)
2. **Étape 2** : Application des délais de publication avec `PublicationDelayTransformer`

Cette séparation offre plusieurs avantages :
- Meilleure modularité et testabilité
- Possibilité de choisir la stratégie de délai (`shift` ou `mask`) indépendamment
- Contrôle fin sur la date de prédiction et le traitement de chaque variable

In [ ]:
# Étape 1 : Imputation des fréquences mixtes (sans imputer les valeurs retardées)
# Initialisation de l'imputer
imputer_step1 = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    keep_lower_frequencies=False,
    cascade_refitting=True,
    imputation_scope='strict',
    delays=delays_df,
    impute_delayed_values=False
)

# Imputation
df_imputed_step1 = imputer_step1.fit_transform(df_timeseries)

# Affichage
print("=== Étape 1 : Imputation des fréquences mixtes ===")
print(f"Shape : {df_imputed_step1.shape}")
print("\n--- Dernières lignes (NaN = valeurs retardées non imputées) ---")
display(df_imputed_step1.tail(6))

In [ ]:
# Étape 2 : Application des délais de publication avec le PublicationDelayTransformer
# Définition des délais au format attendu par le PublicationDelayTransformer
delays_pdt = pd.DataFrame({
    'column': ['inflation_ipc', 'taux_chomage', 'pib_trimestriel', 'balance_commerciale_annuelle'],
    'applicable_delay': [30, 30, 60, 90],
    'unit': ['D', 'D', 'D', 'D'],
    'target_reference_point': ['end', 'end', 'end', 'end']
})

# Création du transformer avec stratégie 'shift'
delay_transformer = PublicationDelayTransformer(
    delays=delays_pdt,
    strategy='shift',
    prediction_date=df_imputed_step1.index.max()
)

# Application des délais sur les données imputées
df_delayed = delay_transformer.fit_transform(df_imputed_step1)

# Affichage
print("=== Étape 2 : Application des délais de publication ===")
print(f"Shape : {df_delayed.shape}")
print("\n--- Dernières lignes après application des délais ---")
display(df_delayed.tail(6))

In [ ]:
# Comparaison avant/après application des délais
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Avant délais
ax1 = axes[0]
for col in df_imputed_step1.columns:
    mask = df_imputed_step1[col].notna()
    ax1.plot(df_imputed_step1.index[mask][-24:], 
             df_imputed_step1[col][mask].tail(24), '-o', markersize=4, label=col)
ax1.set_title("Après imputation (avant délais)", fontsize=11, fontweight='bold')
ax1.legend(fontsize=8)
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, alpha=0.3)

# Après délais
ax2 = axes[1]
for col in df_delayed.columns:
    mask = df_delayed[col].notna()
    ax2.plot(df_delayed.index[mask][-24:], 
             df_delayed[col][mask].tail(24), '-o', markersize=4, label=col)
ax2.set_title("Après application des délais (shift)", fontsize=11, fontweight='bold')
ax2.legend(fontsize=8)
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nLe shift décale les séries dans le temps pour simuler l'information")
print("réellement disponible à la date de prédiction.")

### 7.2 - Intégration dans une validation croisée <a id="72---intégration-dans-une-validation-croisée"></a>

L'intégration du `HighFrequencyImputer` dans une validation croisée temporelle requiert une attention particulière pour éviter tout **leakage temporel**. Voici le workflow recommandé :

```
┌─────────────────────────────────────────────────────────────────────────────────┐
│                    WORKFLOW DE VALIDATION CROISÉE                               │
├─────────────────────────────────────────────────────────────────────────────────┤
│                                                                                 │
│  1. Séparation train / test (splitter temporel)                                 │
│                                                                                 │
│  2. Application des délais de publication sur le train + test                   │
│     └─→ Masquer les valeurs pas encore publiées à la date de coupure            │
│                                                                                 │
│  3. Concaténation train (avec délais) + test (NaN pour y)                       │
│     └─→ Le test a y=NaN pour simuler la prédiction réelle                       │
│                                                                                 │
│  4. Imputation haute fréquence                                                  │
│     └─→ HighFrequencyImputer sur l'ensemble concaténé                           │
│                                                                                 │
│  5. Gestion des délais post-imputation                                          │
│     └─→ PublicationDelayTransformer pour traiter les NaN résiduels              │
│                                                                                 │
│  6. Application de l'horizon de prédiction                                      │
│     └─→ Ajuster y pour refléter l'horizon souhaité (h pas en avant)             │
│                                                                                 │
│  7. (Optionnel) Calcul des variations trimestrielles glissantes                 │
│     └─→ Si y est trimestriel et X est mensuel, recalculer les variations        │
│         trimestrielles en fenêtre glissante pour cohérence inter-fréquences      │
│                                                                                 │
│  8. Entraînement du modèle et évaluation                                        │
│     └─→ Fit sur train, predict sur test, calculer les métriques                 │
│                                                                                 │
└─────────────────────────────────────────────────────────────────────────────────┘
```

Illustrons ce workflow étape par étape.

In [ ]:
# =====================================================================
# Préparation des données
# =====================================================================
# Création d'un jeu de données étendu pour la validation croisée
df_cv = create_timeseries_dataset(start_date='2015-01-01', end_date='2024-06-30', seed=42)

# Variable cible : PIB trimestriel (à prédire)
target_col = 'pib_trimestriel'

# Variables explicatives : les indicateurs mensuels
feature_cols = [c for c in df_cv.columns if c != target_col]

# Affichage
print("=== Données pour la validation croisée ===")
print(f"Période : {df_cv.index.min().strftime('%Y-%m')} à {df_cv.index.max().strftime('%Y-%m')}")
print(f"Variable cible : {target_col}")
print(f"Variables explicatives : {feature_cols}")
print(f"Shape : {df_cv.shape}")

In [ ]:
# =====================================================================
# Étape 1 : Configuration du splitter temporel
# =====================================================================
# Validation croisée out-of-sample avec 3 splits
# Chaque split teste sur 3 mois (1 trimestre)
splitter = TSOutOfSampleSplit(
    n_splits=3,
    test_size=3,
    gap=0  # Pas de gap supplémentaire, les délais de publication s'en chargent
)

# Délais de publication
delays_cv = pd.DataFrame({
    'variable': ['inflation_ipc', 'taux_chomage', 'pib_trimestriel', 'balance_commerciale_annuelle'],
    'applicable_delay': [30, 30, 60, 90],
    'unit': ['D', 'D', 'D', 'D'],
    'target_reference_point': ['end', 'end', 'end', 'end']
})

# Délais au format PublicationDelayTransformer
delays_pdt_cv = pd.DataFrame({
    'column': ['inflation_ipc', 'taux_chomage', 'pib_trimestriel', 'balance_commerciale_annuelle'],
    'applicable_delay': [30, 30, 60, 90],
    'unit': ['D', 'D', 'D', 'D'],
    'target_reference_point': ['end', 'end', 'end', 'end']
})

print("=== Configuration du splitter ===")
print(f"Nombre de splits : {splitter.get_n_splits()}")
print(f"Taille du test : 3 mois (1 trimestre)")

In [ ]:
# =====================================================================
# Boucle de validation croisée complète
# =====================================================================
results = []

for fold_idx, (train_idx, test_idx) in enumerate(splitter.split(df_cv)):
    print(f"\n{'=' * 70}")
    print(f"FOLD {fold_idx + 1}")
    print(f"{'=' * 70}")
    
    # --- Étape 1 : Séparation train / test ---
    df_train_raw = df_cv.iloc[train_idx].copy()
    df_test_raw = df_cv.iloc[test_idx].copy()
    
    print(f"  Train : {df_train_raw.index.min().strftime('%Y-%m')} → "
          f"{df_train_raw.index.max().strftime('%Y-%m')} ({len(df_train_raw)} obs)")
    print(f"  Test  : {df_test_raw.index.min().strftime('%Y-%m')} → "
          f"{df_test_raw.index.max().strftime('%Y-%m')} ({len(df_test_raw)} obs)")
    
    # --- Étape 2 : Application des délais de publication sur le train ---
    # Simulation de l'information disponible à la date de coupure
    prediction_date = df_train_raw.index.max()
    
    delay_transformer_cv = PublicationDelayTransformer(
        delays=delays_pdt_cv,
        strategy='shift',
        prediction_date=prediction_date
    )
    
    df_train_delayed = delay_transformer_cv.fit_transform(df_train_raw)
    
    n_nan_before = df_train_raw.isna().sum().sum()
    n_nan_after = df_train_delayed.isna().sum().sum()
    print(f"  NaN après délais sur train : {n_nan_before} → {n_nan_after}")
    
    # --- Étape 3 : Concaténation train (avec délais) + test (y=NaN) ---
    df_test_for_concat = df_test_raw.copy()
    # Mise à NaN de la cible dans le test pour simuler la prédiction réelle
    df_test_for_concat[target_col] = np.nan
    
    df_concat = pd.concat([df_train_delayed, df_test_for_concat], axis=0)
    df_concat = df_concat.sort_index()
    
    # --- Étape 4 : Imputation haute fréquence ---
    imputer_cv = HighFrequencyImputer(
        target_frequency='M',
        estimator=LinearRegression(),
        keep_lower_frequencies=False,
        cascade_refitting=True,
        imputation_scope='strict',
        delays=delays_cv,
        impute_delayed_values=False
    )
    
    df_imputed_cv = imputer_cv.fit_transform(df_concat)
    print(f"  Shape après imputation : {df_imputed_cv.shape}")
    
    # --- Étape 5 : Gestion des délais post-imputation ---
    delay_post = PublicationDelayTransformer(
        delays=delays_pdt_cv,
        strategy='shift',
        prediction_date=prediction_date
    )
    df_post_delay = delay_post.fit_transform(df_imputed_cv)
    
    # --- Étape 6 : Calcul des variations trimestrielles glissantes ---
    # Pour chaque variable mensuelle, calcul de la variation sur 3 mois
    # afin d'être cohérent avec la fréquence trimestrielle de la cible
    df_quarterly_vars = pd.DataFrame(index=df_post_delay.index)
    
    for col in feature_cols:
        if col in df_post_delay.columns:
            # Variation sur 3 mois (fenêtre glissante trimestrielle)
            df_quarterly_vars[f'{col}_var3m'] = df_post_delay[col].pct_change(periods=3) * 100
    
    # Ajout de la cible (variation trimestrielle du PIB)
    df_quarterly_vars[target_col] = df_post_delay[target_col]
    df_quarterly_vars[f'{target_col}_var'] = (
        df_post_delay[target_col].pct_change(periods=3) * 100
    )
    
    # Conservation des variables originales aussi
    for col in feature_cols:
        if col in df_post_delay.columns:
            df_quarterly_vars[col] = df_post_delay[col]
    
    # --- Étape 7 : Préparation des features et de la cible ---
    # Sélection des features : variations trimestrielles des variables mensuelles
    var3m_cols = [c for c in df_quarterly_vars.columns if c.endswith('_var3m')]
    
    # Filtrage sur les périodes de test
    y_target = f'{target_col}_var'
    
    # Séparation train/test dans le DataFrame imputé
    df_train_final = df_quarterly_vars.loc[df_train_raw.index].dropna(subset=var3m_cols + [y_target])
    df_test_final = df_quarterly_vars.loc[df_test_raw.index]
    
    # Évaluation sur les observations trimestrielles du test
    # (seuls les mois de fin de trimestre ont des valeurs PIB non-NaN dans les données originales)
    test_quarterly_mask = df_test_raw[target_col].notna()
    
    if test_quarterly_mask.any() and len(df_train_final) > 0:
        X_train = df_train_final[var3m_cols]
        y_train = df_train_final[y_target]
        
        test_dates = df_test_raw.index[test_quarterly_mask]
        X_test = df_quarterly_vars.loc[test_dates, var3m_cols]
        y_test_true = df_test_raw.loc[test_dates, target_col].pct_change(periods=3).dropna() * 100
        
        # Alignement des index
        common_idx = X_test.index.intersection(y_test_true.index)
        
        if len(common_idx) > 0 and not X_test.loc[common_idx].isna().any().any():
            X_test_aligned = X_test.loc[common_idx]
            y_test_aligned = y_test_true.loc[common_idx]
            
            # --- Étape 8 : Entraînement et évaluation ---
            model = LinearRegression()
            model.fit(X_train.fillna(0), y_train.fillna(0))
            y_pred = model.predict(X_test_aligned.fillna(0))
            
            rmse = np.sqrt(mean_squared_error(y_test_aligned, y_pred))
            mae = mean_absolute_error(y_test_aligned, y_pred)
            
            results.append({
                'fold': fold_idx + 1,
                'train_size': len(df_train_final),
                'test_size': len(common_idx),
                'rmse': rmse,
                'mae': mae
            })
            
            print(f"  Résultats : RMSE={rmse:.4f}, MAE={mae:.4f}")
        else:
            print(f"  Pas assez de données alignées pour l'évaluation")
    else:
        print(f"  Pas de données trimestrielles dans le test")

In [ ]:
# =====================================================================
# Synthèse des résultats de la validation croisée
# =====================================================================
if results:
    df_results = pd.DataFrame(results)
    
    print("\n" + "=" * 70)
    print("SYNTHÈSE DE LA VALIDATION CROISÉE")
    print("=" * 70)
    display(df_results)
    
    print(f"\n--- Métriques moyennes ---")
    print(f"  RMSE moyen : {df_results['rmse'].mean():.4f} (± {df_results['rmse'].std():.4f})")
    print(f"  MAE moyen  : {df_results['mae'].mean():.4f} (± {df_results['mae'].std():.4f})")
else:
    print("Aucun résultat de validation croisée disponible.")
    print("Cela peut arriver si les données de test ne contiennent pas")
    print("d'observations trimestrielles complètes.")

#### Points clés du workflow de validation croisée

**Prévention du leakage temporel** : le workflow ci-dessus applique les délais de publication sur le train *avant* l'imputation. Ainsi, les modèles d'imputation ne voient jamais d'information qui ne serait pas disponible à la date de coupure.

**Cohérence inter-fréquences** : lorsque la variable cible est trimestrielle (PIB) et que les features sont mensuelles, le calcul de variations trimestrielles en fenêtre glissante (`pct_change(periods=3)`) permet de comparer des variations sur des horizons cohérents. Chaque mois dispose ainsi d'une "variation trimestrielle glissante" qui peut être mise en relation avec la croissance du PIB.

**Ordre des opérations** : l'enchaînement délais → imputation → délais post-imputation → horizon → variations est crucial. Toute inversion pourrait introduire un biais ou du leakage.

```
  ┌─────────────┐   ┌──────────────┐   ┌──────────────┐   ┌──────────────┐   ┌────────────────┐
  │   Délais    │   │  Imputation  │   │  Délais      │   │  Variations  │   │  Modèle de     │
  │   (train)   │──▶│  (HF)       │──▶│  post-imput. │─▶│  Q glissant   │─▶│  prévision     │
  └─────────────┘   └──────────────┘   └──────────────┘   └──────────────┘   └────────────────┘
```

---

## 8 - Résumé et tableau récapitulatif

<a id="8---résumé-et-tableau-récapitulatif"></a>

### 8.1 - Tableau récapitulatif des paramètres

<a id="81---tableau-récapitulatif-des-paramètres"></a>

| Paramètre | Type | Défaut | Description | Impact |
|-----------|------|--------|-------------|--------|
| `target_frequency` | `str \| Dict` | - | Fréquence cible pour l'imputation (ex: `"M"`, `"Q"`) | Détermine la granularité de sortie |
| `estimator` | `Estimator \| Dict` | - | Modèle(s) pour prédire les valeurs manquantes | Qualité des imputations |
| `keep_lower_frequencies` | `bool` | `True` | Conserver les fréquences intermédiaires | Structure de sortie (MultiIndex avec la fréquence en avant-dernier niveau ou Index simple) |
| `cascade_refitting` | `bool` | `True` | Réentraîner après chaque niveau de fréquence | Précision vs Vitesse |
| `imputation_scope` | `Literal` | `'strict'` | Extension de la fenêtre d'imputation pour l'entraînement | Taille du jeu d'entraînement |
| `attrition_threshold` | `float [0-1]` | `0.5` | Seuil minimum de colonnes disponibles | Étendue de la fenêtre d'entraînement |
| `train_on_partial_coverage` | `bool` | `False` | Utiliser les valeurs imputées pour l'entraînement | Qualité vs Quantité de données |
| `impute_delayed_values` | `bool` | `False` | Imputer les valeurs affectées par des délais de publication | Gestion des données récentes |
| `delays` | `DataFrame \| None` | `None` | Jeu de données des délais de publication à tenir compte | Précision du calcul de la période d'imputation |

### 8.2 - Recommandations par cas d'usage

<a id="82---recommandations-par-cas-dusage"></a>

| Cas d'usage | `keep_lower_frequencies` | `cascade_refitting` | `imputation_scope` | `attrition_threshold` | `train_on_partial_coverage` | Justification |
|-------------|:-----------------------:|:-------------------:|:------------------:|:--------------------:|:--------------------------:|---------------|
| 🚀 Prototypage rapide | `False` | `False` | `strict` | 0.5 | `False` | Configuration minimale, exécution rapide |
| 📊 Reporting multi-fréquence | `True` | `False` | `strict` | 0.5 | `False` | Conservation des niveaux pour analyse à plusieurs fréquences |
| 🎯 Production haute précision | `False` | `True` | `extended_both` | 0.5 | `False` | Cascade complète avec extension de fenêtre |
| 📈 Données récentes avec délais | `False` | `True` | `extended_forward` | 0.4 | `True` | Extension vers le futur pour données récentes manquantes |
| 🗄️ Données historiques limitées | `True` | `True` | `extended_backward` | 0.3 | `True` | Seuil bas et extension vers le passé |

### 8.3 - Points clés à retenir

<a id="83---points-clés-à-retenir"></a>

#### Points clés à retenir

**1. Structure de sortie**

| Paramètre | Comportement |
|-----------|-------------|
| `keep_lower_frequencies=False` | Index simple, une seule fréquence |
| `keep_lower_frequencies=True` | MultiIndex avec toutes les fréquences |

**2. Stratégie d'imputation**

| Paramètre | Comportement |
|-----------|-------------|
| `cascade_refitting=False` | Un seul entraînement, imputation directe |
| `cascade_refitting=True` | Cascade avec réentraînement, plus précis |

**3. Fenêtre d'entraînement**

- **Période d'imputation stricte** = Période où toutes les séries ont des vraies valeurs
- `imputation_scope` étend P1 (before/after/both) selon `attrition_threshold`
- `train_on_partial_coverage` inclut ou non les valeurs imputées

**4. Traçabilité (Provenance)**

| Type | Description |
|------|-------------|
| `ORIGINAL` | Valeurs originales non modifiées |
| `MODEL_ON_TRUE` | Imputées par modèle entraîné sur vraies valeurs |
| `MODEL_ON_MIXED` | Imputées par modèle entraîné sur valeurs mixtes |
| `AGGREGATED` | Obtenues par agrégation temporelle |

**5. Bonnes pratiques**

- Commencer par une configuration simple (Scénario A)
- Augmenter progressivement la complexité si nécessaire
- Toujours vérifier la matrice de provenance
- Valider les résultats sur un échantillon de test
- Documenter les paramètres utilisés pour la reproductibilité

---

#### Arbre de décision pour le choix des paramètres

```
                               ┌─────────────────────────────┐
                               │  Besoin de toutes les       │
                               │  fréquences en sortie ?     │
                               └─────────────┬───────────────┘
                                             │
                         ┌───────────────────┴───────────────────┐
                         │                                       │
                        OUI                                     NON
                         │                                       │
                         ▼                                       ▼
           ┌─────────────────────────┐           ┌─────────────────────────┐
           │ keep_lower_frequencies  │           │ keep_lower_frequencies  │
           │        = True           │           │        = False          │
           └───────────┬─────────────┘           └───────────┬─────────────┘
                       │                                     │
                       └─────────────────┬───────────────────┘
                                         │
                                         ▼
                       ┌─────────────────────────────────────┐
                       │ Présence de délai de publication ?  │
                       └─────────────┬───────────────────────┘
                                     │
                   ┌─────────────────┴─────────────────┐
                   │                                   │
                  OUI                                 NON
                   │                                   │
                   ▼                                   ▼
      ┌────────────────────────┐         ┌────────────────────────┐
      │ impute_delayed_values  │         │ impute_delayed_values  │
      │      = False           │         │      = True            │
      │                        │         │                        │
      │ imputation_scope =     │         │ imputation_scope =     │
      │ "strict" ou            │         │ "strict", "extended_   │
      │ "extended_backward"    │         │ backward", "extended_  │
      └──────────┬─────────────┘         │ forward" ou "extended_ │
                 │                       │ both"                  │
                 │                       └──────────┬─────────────┘
                 │                                  │
                 └─────────────┬────────────────────┘
                               │
                               ▼
           ┌─────────────────────────────────────────────┐
           │ Période stricte contient peu                │
           │ d'observations ET qualité des imputations   │
           │ très élevée ?                               │
           └─────────────────┬───────────────────────────┘
                             │
             ┌───────────────┴───────────────┐
             │                               │
            OUI                             NON
             │                               │
             ▼                               ▼
    ┌────────────────────┐        ┌──────────────────────────┐
    │ imputation_scope = │        │ Beaucoup de séries       │
    │ "extended_both" ou │        │ disponibles avant le     │
    │ "extended_forward" │        │ début de période stricte?│
    └────────────────────┘        └──────────┬───────────────┘
                                             │
                           ┌─────────────────┴─────────────────┐
                           │                                   │
                          OUI                                 NON
                           │                                   │
                           ▼                                   ▼
              ┌────────────────────────┐         ┌────────────────────────┐
              │ imputation_scope =     │         │ imputation_scope =     │
              │ "extended_backward" ou │         │ "strict"               │
              │ "extended_both"        │         │                        │
              └────────┬───────────────┘         └──────────┬─────────────┘ 
                       │                                    │
                       └───────────────┬────────────────────┘
                                       │
                                       ▼
                 ┌─────────────────────────────────────────────┐
                 │ Imputations à fréquence intermédiaire       │
                 │ de grande qualité ?                         │
                 └─────────────────┬───────────────────────────┘
                                   │
                 ┌─────────────────┴─────────────────┐
                 │                                   │
                OUI                                 NON
                 │                                   │
                 ▼                                   ▼
    ┌────────────────────────┐         ┌────────────────────────┐
    │ cascade_refitting =    │         │ cascade_refitting =    │
    │ True                   │         │ False                  │
    └────────┬───────────────┘         └────────┬───────────────┘
             │                                  │
             └────────────────┬─────────────────┘
                              │
                              ▼
         ┌─────────────────────────────────────────────────────┐
         │ Covariables fortement corrélées entre elles OU      │
         │ peu de séries contiennent beaucoup d'information ?  │
         └─────────────────┬───────────────────────────────────┘
                           │
         ┌─────────────────┴─────────────────┐
         │                                   │
        OUI                                 NON
         │                                   │
         ▼                                   ▼
┌────────────────────┐           ┌────────────────────┐
│ attrition_threshold│           │ attrition_threshold│
│ plus petit         │           │ plus grand         │
│ (ex: 0.3-0.5)      │           │ (ex: 0.6-0.8)      │
└────────────────────┘           └────────────────────┘
```

## Conclusion

Ce notebook a présenté de manière détaillée le fonctionnement de la classe `HighFrequencyImputer` pour l'imputation de données à fréquences mixtes. Les points essentiels à retenir sont :

1. **La fenêtre d'imputation** est centrale dans le processus : c'est la période sur laquelle les valeurs vont être imputées à des fréquences différentes. Du nombre d'observations disponibles dépendra la qualité de l'imputation.

2. **Les paramètres clés** (`keep_lower_frequencies`, `cascade_refitting`, `imputation_scope`, `attrition_threshold`) permettent un contrôle fin du processus d'imputation.

3. **La matrice de provenance** permet de tracer l'origine de chaque valeur (originale, imputée par modèle sur vraies valeurs, imputée par modèle sur valeurs mixtes, ou agrégée).

4. **Les délais de publication** sont gérés via les paramètres `delays` et `impute_delayed_values`, avec la possibilité d'inférer les délais automatiquement ou de les spécifier explicitement.

5. **L'intégration dans un workflow** de validation croisée nécessite un enchaînement rigoureux : séparation train/test, application des délais, imputation, gestion post-imputation, et calcul des variations cohérentes entre fréquences.

6. **Le choix des paramètres** dépend du cas d'usage : prototypage rapide, production haute précision, ou recherche complète.